In [ ]:
# ================================================================
# НЕЙРОСЕТЕВОЙ OFDM-ПРИЁМНИК
# Оценка канала и детектирование сигнала
#
# СТРУКТУРА НОУТБУКА
# ──────────────────
# Блок 0  — Настройка среды (переменные окружения, GPU, seed)
# Блок 1  — Параметры OFDM-системы, канала, обучения
# Блок 2  — Параметры модуляций и флаги управления
# Блок 3  — Классы модуляций, модель канала, OFDM-генератор
# Блок 4  — Архитектуры нейросетей (ChanEstNet, DetectorNet, JointNet)
# Блок 5  — Функции обучения (три этапа + классическое)
# Блок 6  — Классические и нейросетевые детекторы, вычисление BER
# Блок 7  — Построение BER-кривых и диаграмм созвездий
# Блок 8  — Запуск полного эксперимента
# ================================================================

In [ ]:
# ================================================================
# БЛОК 0: НАСТРОЙКА СРЕДЫ
# ================================================================
#
# Задача блока: подготовить вычислительную среду к корректной
# и воспроизводимой работе. Все переменные окружения обязательно
# устанавливать ДО первого импорта TensorFlow — иначе не подействуют.
# ================================================================

import os

# Подавляем TensorFlow INFO/WARNING сообщения (оставляем только ошибки).
# Уровни: 0=все, 1=warning+error, 2=только error, 3=ничего.
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# Отключаем оптимизации oneDNN (Intel MKL-DNN). Включённые оптимизации
# могут давать незначительно разные результаты из-за порядка операций
# с плавающей точкой (fused ops) — отключаем для полной воспроизводимости.
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

# «Зерно» для генераторов случайных чисел.
# При одинаковом SEED все «случайные» числа генерируются в одном и том
# же порядке при каждом запуске → эксперименты воспроизводимы.
SEED = 42

# Фиксируем хэш-функцию Python. Без этого словари и множества могут
# обходить элементы в разном порядке между запусками процесса,
# что влияет на воспроизводимость некоторых операций NumPy.
os.environ["PYTHONHASHSEED"] = str(SEED)

print("Блок 0 готов.")

In [ ]:
#================================================================
# БЛОК 1
# ПАРАМЕТРЫ OFDM-СИСТЕМЫ, КАНАЛА И НЕЙРОСЕТЕВОГО ЭКСПЕРИМЕНТА
# ================================================================
#
# Этот блок — «центр управления» всего кода.
# Здесь задаются все числовые параметры системы.
# Все остальные блоки только ЧИТАЮТ эти переменные, не изменяя их.
# ================================================================

import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers as KL

import gc

# Очищаем Keras-сессию и запускаем сборщик мусора перед стартом.
# Важно при повторном запуске ячеек Jupyter: уничтожает старые TF-графы
# из предыдущего прогона и освобождает место под новые.
tf.keras.backend.clear_session()
gc.collect()

# ── Подавление предупреждений (после импорта TF) ─────────────────
# Применяется и к Python-модулю logging, который TF использует внутри.
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import logging
logging.getLogger('tensorflow').setLevel(logging.ERROR)

# Autograph превращает Python-функции в TF-графы и при перекомпиляции
# выводит verbose-предупреждения. Уровень 0 их подавляет.
try:
    tf.autograph.set_verbosity(0)
except Exception:
    pass
# ─────────────────────────────────────────────────────────────────


# ================================================================
# ВОСПРОИЗВОДИМОСТЬ ЭКСПЕРИМЕНТОВ
# ================================================================
#
# Фиксируем ВСЕ генераторы случайных чисел:
#   • np.random — генерация данных (биты, шум, канал)
#   • tf.random — инициализация весов сетей
#
# Без фиксации невозможно отличить «улучшение от новой архитектуры»
# от «удачного случайного старта».
# ================================================================

np.random.seed(SEED)
tf.random.set_seed(SEED)


# ================================================================
# НАСТРОЙКА GPU
# ================================================================
#
# По умолчанию TF при старте резервирует ВСЮ видеопамять.
# set_memory_growth=True переключает на ленивое выделение:
# память занимается только по мере реальной потребности.
# ================================================================

for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)


# ================================================================
# МОДЕЛЬ НЕЛИНЕЙНОГО УСИЛИТЕЛЯ МОЩНОСТИ (POWER AMPLIFIER, PA)
# ================================================================
#
# Реальный усилитель нелинеен: при больших амплитудах «насыщается»
# и искажает форму сигнала. Для OFDM с высоким PAPR (пик-фактором)
# это принципиальная проблема.
#
# Используется модель Rapp (AM/AM без AM/PM):
#   amp_out = amp_in / (1 + (amp_in/A_sat)^(2p))^(1/2p)
#
# USE_PA_NONLINEARITY = False → линейный усилитель, пилоты чистые,
#   классическая LS-оценка канала работает корректно.
# USE_PA_NONLINEARITY = True  → пилоты искажены PA, LS-оценка
#   деградирует — потенциальное преимущество для DNN-подходов.
#
# PA_IBO_DB — Input Back-Off в дБ: рабочая точка относительно
#   точки насыщения. Меньше → сильнее нелинейность.
#   2–3 дБ — типичный жёсткий режим IoT-передатчиков.
# PA_RAPP_P — сглаживание перехода в насыщение (большие p → резче).
# ================================================================

USE_PA_NONLINEARITY = True
PA_IBO_DB           = 1.0
PA_RAPP_P           = 5.0


# ================================================================
# РЕЖИМ ИССЛЕДОВАНИЯ (RESEARCH_MODE)
# ================================================================
#
# Главный переключатель: определяет задачу нейросети,
# формат входа/выхода и функцию потерь при обучении.
#
# CHANNEL_ESTIMATION
#   Нейросеть оценивает канал: Y_pilot_norm → H_data_norm.
#   Loss: MSE. После — классическая ZF/MMSE-эквализация.
#
# SIGNAL_DETECTION
#   Нейросеть детектирует символы по уже эквализованным данным.
#   Вход: ZF-символы Z = Y_data/H_est. Loss: BCE.
#   Здесь DNN реально выигрывает у hard-детектора на низких SNR
#   за счёт «мягкого» решения у краёв созвездия.
#
# END_TO_END
#   Нейросеть делает всё сама по «сырому» OFDM-символу.
#   Вход: Re(Y_freq), Im(Y_freq) (все поднесущие). Loss: BCE.
#   Самая тяжёлая задача: сеть учится и оценивать H, и детектировать.
# ================================================================

RESEARCH_MODE = "END_TO_END"

# Возможные варианты (раскомментируй нужный):
# RESEARCH_MODE = "CHANNEL_ESTIMATION"
# RESEARCH_MODE = "SIGNAL_DETECTION"
# RESEARCH_MODE = "END_TO_END"


# ================================================================
# МОДЕЛЬ КАНАЛА (CHANNEL_MODEL)
# ================================================================
#
# RAYLEIGH — многолучевой канал без прямой видимости (NLOS).
#   Амплитуда каждого отвода ~ Rayleigh. Типично для города.
#
# RICIAN — канал с прямой видимостью (LOS).
#   Один сильный прямой луч + случайные отражения.
#   Соотношение LOS/NLOS задаётся K_FACTOR_DB.
#
# EPA — Extended Pedestrian A (3GPP TS 36.101, Annex B).
#   7 отводов, быстроубывающий профиль. Слабая частотная
#   избирательность. Модель пешехода.
#
# EVA — Extended Vehicular A (3GPP).
#   9 отводов, большие задержки. Высокая частотная
#   избирательность. Модель автомобиля.
#
# ETU — Extended Typical Urban (3GPP).
#   9 отводов, практически равномерная мощность первых 6 отводов.
#   Самый сложный из 3GPP-профилей для LS-оценки канала.
# ================================================================

CHANNEL_MODEL = "ETU"

# Возможные варианты:
# CHANNEL_MODEL = "RAYLEIGH"
# CHANNEL_MODEL = "RICIAN"
# CHANNEL_MODEL = "EPA"
# CHANNEL_MODEL = "EVA"
# CHANNEL_MODEL = "ETU"


# K-фактор Райса. Используется ТОЛЬКО при CHANNEL_MODEL = "RICIAN".
# K_lin = 10^(K_dB/10). При K_lin: LOS=K/(K+1), NLOS=1/(K+1).
# 0 dB → почти Rayleigh; 6 dB → LOS в 4× мощнее NLOS; 10 dB → в 10×.
K_FACTOR_DB = 6.0


# ================================================================
# ПРОФИЛИ КАНАЛОВ (CHANNEL_PROFILES)
# ================================================================
#
# N_TAPS — число отводов дискретной модели канала.
# Физический смысл: сигнал приходит несколькими путями (прямой +
# отражения). Каждый путь = один отвод.
# Дискретная свёртка: y[n] = sum_k h[k]*x[n-k].
#
# Конкретные мощности отводов для EPA/EVA/ETU задаются в Блоке 3
# через таблицу _PDP_DB (3GPP-стандартные профили задержек-мощностей).
# ================================================================

CHANNEL_PROFILES = {
    "RAYLEIGH": {"N_TAPS": 4},
    "RICIAN":   {"N_TAPS": 4},
    "EPA":      {"N_TAPS": 7},
    "EVA":      {"N_TAPS": 9},
    "ETU":      {"N_TAPS": 9},
}


# ================================================================
# СЦЕНАРИИ ЭКСПЕРИМЕНТОВ (EXPERIMENTS)
# ================================================================
#
# Каждый сценарий — набор параметров OFDM-системы и диапазонов SNR.
# Активный сценарий выбирается переменной SCENARIO ниже.
#
# ВАЖНО: конкретные значения N_FFT, N_CP, N_PILOT и диапазоны SNR
# активно меняются в процессе исследования. Описания ниже отражают
# НАЗНАЧЕНИЕ сценариев, а не жёсткие конфигурации.
#
# BASELINE — базовый эталон для первичной проверки работоспособности.
#   Умеренное N_FFT, достаточное число пилотов (≥25%), стандартный
#   SNR-диапазон. Классические методы работают хорошо — отправная точка.
#
# CHALLENGING — затрудняет классические методы.
#   Может включать: малое число пилотов, широкий SNR-диапазон,
#   другие условия, где LS/MMSE деградируют.
#
# DNN_FRIENDLY — условия, где DNN потенциально превосходит классику.
#   Может включать: большое N_FFT, малое число пилотов (классика
#   теряет точность при длинном H и редких пилотах), широкий SNR.
# ================================================================

EXPERIMENTS = {
    "BASELINE": {
        "N_FFT":         64,
        "N_CP":          16,
        "N_PILOT":       16,   # 25% пилотов
        # "SNR_TRAIN_MIN": -5,
        # "SNR_TRAIN_MAX": 25,
        # "SNR_TEST_MIN":  -10,
        # "SNR_TEST_MAX":  25,
        "SNR_TRAIN_MIN": -20,
        "SNR_TRAIN_MAX": 60,
        "SNR_TEST_MIN":  -10,
        "SNR_TEST_MAX":  40,
    },
    "CHALLENGING": {
        "N_FFT":         64,
        "N_CP":          16,
        "N_PILOT":       16,
        "SNR_TRAIN_MIN": -10,
        "SNR_TRAIN_MAX": 26,
        "SNR_TEST_MIN":  -10,
        "SNR_TEST_MAX":  40,
    },
    "DNN_FRIENDLY": {
        "N_FFT":         128,
        "N_CP":          32,
        "N_PILOT":       8,    
        # "SNR_TRAIN_MIN": -20,
        # "SNR_TRAIN_MAX": 60,
        # "SNR_TEST_MIN":  -10,
        # "SNR_TEST_MAX":  40,
        "SNR_TRAIN_MIN": -5,
        "SNR_TRAIN_MAX": 25,
        "SNR_TEST_MIN":  -10,
        "SNR_TEST_MAX":  25,
    },
}

SCENARIO = "DNN_FRIENDLY"

# Возможные варианты:
# SCENARIO = "BASELINE"
# SCENARIO = "CHALLENGING"
# SCENARIO = "DNN_FRIENDLY"

# Загружаем параметры выбранного сценария.
CFG = EXPERIMENTS[SCENARIO]


# ================================================================
# OFDM-ПАРАМЕТРЫ
# ================================================================

# N_FFT — число поднесущих (размер FFT). Должно быть степенью 2.
N_FFT = CFG["N_FFT"]

# N_CP — длина циклического префикса (Cyclic Prefix).
# CP — копия хвоста OFDM-символа, добавляемая в его начало.
# Поглощает межсимвольные помехи (ISI) при многолучевом распространении,
# превращая линейную свёртку в циклическую. Это позволяет FFT-приёмнику
# получить диагональный канал: Y[k] = H[k]*X[k] + N[k] (нет ISI!).
# Критическое условие: N_CP >= N_TAPS - 1 (проверяется assert ниже).
N_CP = CFG["N_CP"]

# N_PILOT — число пилотных поднесущих.
# Пилоты — заранее известные обеим сторонам символы (X_pilot = 1+0j).
# Из Y_pilot = H_pilot + шум приёмник оценивает H_pilot ≈ Y_pilot.
# Больше пилотов → точнее оценка H, но меньше места для данных.
N_PILOT = CFG["N_PILOT"]

# N_DATA — информационные поднесущие, несущие реальные данные.
N_DATA = N_FFT - N_PILOT

N_TAPS = CHANNEL_PROFILES[CHANNEL_MODEL]["N_TAPS"]


# ================================================================
# ПАРАМЕТРЫ ОБУЧЕНИЯ
# ================================================================
#
# N_TRAIN: число OFDM-символов для обучения (1 символ = 1 выборка).
#   200_000 — для быстрых проверок; 1_000_000 — для итоговых.
#
# N_TEST: символов для одной точки BER-кривой.
#   При QPSK: N_TEST × N_DATA × 2 бит на точку SNR.
#
# BATCH_SIZE: размер мини-батча. Больше → стабильнее градиент
#   + лучше утилизация GPU. LR масштабируется с batch (см. _SCALED_LR).
# ================================================================

N_TRAIN    = 1_000_000//10
N_TEST     = 10_000
BATCH_SIZE = 2048*2   


# ================================================================
# ДИАПАЗОНЫ SNR
# ================================================================
#
# КЛЮЧЕВОЙ ПРИНЦИП: сеть обучается на ВСЁМ диапазоне SNR сразу
# (не только на высоких значениях). Это обеспечивает хорошее
# обобщение на любые условия шума.
#
# SNR_TEST — точки для BER-кривой; шаг 1 дБ даёт плавную кривую.
# ================================================================

SNR_TRAIN_MIN = CFG["SNR_TRAIN_MIN"]
SNR_TRAIN_MAX = CFG["SNR_TRAIN_MAX"]

SNR_TEST = np.arange(
    CFG["SNR_TEST_MIN"],
    CFG["SNR_TEST_MAX"] + 1,
    1
)




# USE_SNR_FEATURE: передаём нейросети нормированный SNR как признак.
# Позволяет одной модели адаптировать стратегию к разным уровням шума,
# а не усреднять поведение. Даёт ощутимый выигрыш в BER при широком SNR.
USE_SNR_FEATURE = True

# USE_INPUT_NORMALIZATION: нормируем вход на RMS-амплитуду пилотов:
#   norm = sqrt(mean(|Y_pilot|²)).
# Убирает зависимость масштаба входа от SNR — вход всегда ~O(1),
# что упрощает обучение и снижает чувствительность к инициализации.
USE_INPUT_NORMALIZATION = False



# ================================================================
# РАЗМЕЩЕНИЕ ПИЛОТОВ И ИНФОРМАЦИОННЫХ ПОДНЕСУЩИХ
# ================================================================
#
# РАВНОМЕРНОЕ размещение пилотов — обязательное условие для
# DFT-based оценщика из Блока 3. При равномерном шаге D = N_FFT/N_PILOT
# IFFT пилотных наблюдений даёт импульсную характеристику без алиасинга
# (если N_TAPS <= N_PILOT). Assert гарантирует целочисленность шага.
# ================================================================

assert N_FFT % N_PILOT == 0, \
    f"N_FFT={N_FFT} должен делиться на N_PILOT={N_PILOT} без остатка."

# Пример: N_FFT=64, N_PILOT=16 → шаг=4 → [0, 4, 8, ..., 60]
PILOT_IDX = np.arange(0, N_FFT, N_FFT // N_PILOT)

# Все поднесущие кроме пилотных
DATA_IDX = np.array([k for k in range(N_FFT) if k not in PILOT_IDX])



# ================================================================
# КОНФИГУРАЦИЯ АРХИТЕКТУРЫ НЕЙРОСЕТЕЙ
# ================================================================
# Позволяет настраивать глубину и ширину сетей из одного места.
# Все функции build_*_net() читают эти параметры.
#
# conv_layers: список кортежей (filters, kernel_size)
# dense_layers: список размеров Dense-слоёв
#
# ================================================================
ARCH_CONFIG = {
    'ChanEstNet': {
        'conv_layers': [(64, 3), (128, 3), (128, 3), (64, 3)],
        'dense_layers': [512, 512, 256],
    },
    'DetectorNet': {
        'conv_layers': [(64, 1), (128, 3), (128, 3), (128, 3), (64, 1)],
    },
    'JointNet': {
        'est_conv_layers': [(64, 3), (128, 3), (128, 3), (64, 3)],
        'est_dense_layers': [512, 512, 256],
        # ВОЗВРАЩАЕМ СВЁРТКИ: критично для интерполяции при разреженных пилотах
        'det_conv_layers': [(64, 1), (128, 3), (128, 3), (128, 3), (64, 1)],
    },
}

# ================================================================
# ПАРАМЕТРЫ ОБУЧЕНИЯ JOINTNET
# ================================================================
# Вариант 2: False = обучать JointNet с нуля (без переноса весов)
# Это позволяет сети найти совершенно новую стратегию для
# нелинейного канала, не ограниченную предварительным обучением.
JOINT_TRANSFER_WEIGHTS = True

# Увеличенное число эпох для обучения с нуля
JOINT_EPOCHS = 150      # было 80
JOINT_TRAIN_SAMPLES = N_TRAIN  # было N_TRAIN // 2 — используем все данные



# ================================================================
# КРИТИЧЕСКАЯ ПРОВЕРКА: N_CP >= N_TAPS
# ================================================================
#
# Если N_CP < N_TAPS, каждый OFDM-символ «загрязняет» следующий (ISI).
# Формула Y[k]=H[k]*X[k]+N[k] ломается, OFDM перестаёт работать.
# ================================================================

assert N_CP >= N_TAPS, (
    f"Циклический префикс (N_CP={N_CP}) короче "
    f"импульсной характеристики (N_TAPS={N_TAPS})! "
    f"Увеличь N_CP или выбери другой сценарий/канал."
)

print("=" * 65)
print("КОНФИГУРАЦИЯ ЭКСПЕРИМЕНТА")
print("=" * 65)
print(f"Режим исследования      : {RESEARCH_MODE}")
print(f"Модель канала           : {CHANNEL_MODEL}")
if CHANNEL_MODEL == "RICIAN":
    print(f"K-factor                : {K_FACTOR_DB:.1f} dB")
print(f"N_FFT={N_FFT}  N_CP={N_CP}  N_PILOT={N_PILOT}  N_DATA={N_DATA}  N_TAPS={N_TAPS}")
print(f"N_TRAIN={N_TRAIN:,}  N_TEST={N_TEST:,}  BATCH_SIZE={BATCH_SIZE:,}")
print(f"SNR TRAIN: {SNR_TRAIN_MIN}…{SNR_TRAIN_MAX} dB   TEST: {SNR_TEST[0]}…{SNR_TEST[-1]} dB")
gpu_list = tf.config.list_physical_devices("GPU")



# ================================================================
# АВТОМАТИЧЕСКОЕ УПРАВЛЕНИЕ ПАПКАМИ (ВСЕ ФАЙЛЫ ЭКСПЕРИМЕНТА)
# ================================================================
def generate_experiment_dir_name():
    """
    Генерирует уникальное имя папки на основе ВСЕХ параметров,
    влияющих на архитектуру нейросети и эксперимент.
    """
    # Базовые параметры
    parts = [
        f"EXP_{RESEARCH_MODE}",
        f"{CHANNEL_MODEL}",
        f"{SCENARIO}",
        f"P{N_PILOT}",
        f"BS{BATCH_SIZE}",
        f"SNR{SNR_TRAIN_MIN}_{SNR_TRAIN_MAX}",
    ]
    
    # Параметры нелинейности (если включена)
    if USE_PA_NONLINEARITY:
        parts.append(f"PA_IBO{int(PA_IBO_DB)}")
        parts.append(f"RAPP_P{PA_RAPP_P}")
    
    # Параметры архитектуры (влияют на структуру нейросети)
    if RESEARCH_MODE == "END_TO_END":
        # Для JointNet добавляем параметры детектора
        if 'det_dense_layers' in ARCH_CONFIG.get('JointNet', {}):
            det_layers = ARCH_CONFIG['JointNet']['det_dense_layers']
            parts.append(f"DetDense{'_'.join(map(str, det_layers))}")
        elif 'det_conv_layers' in ARCH_CONFIG.get('JointNet', {}):
            det_layers = ARCH_CONFIG['JointNet']['det_conv_layers']
            parts.append(f"DetConv{'_'.join([f'{f}x{k}' for f,k in det_layers])}")
    
    # Параметры обучения JointNet
    if not JOINT_TRANSFER_WEIGHTS:
        parts.append("TrainFromScratch")
    parts.append(f"Epochs{JOINT_EPOCHS}")
    
    # Дополнительные признаки
    if USE_SNR_FEATURE:
        parts.append("WithSNR")
    if not USE_INPUT_NORMALIZATION:
        parts.append("NoNorm")
    
    return "_".join(parts)

# Генерируем имя папки
EXP_DIR_NAME = generate_experiment_dir_name()
BASE_SAVE_DIR = EXP_DIR_NAME
os.makedirs(BASE_SAVE_DIR, exist_ok=True)

print(f" Директория эксперимента: '{BASE_SAVE_DIR}/'")
print(f"TF {tf.__version__}  GPU: {gpu_list[0].name if gpu_list else 'CPU mode'}")
print("\n✓ Блок 1 успешно выполнен")

In [ ]:
# ================================================================
# БЛОК 2
# ПАРАМЕТРЫ МОДУЛЯЦИИ И СОЗВЕЗДИЙ
# ================================================================
#
# Модуляция — кодирование битов в комплексные символы (точки I/Q)
# для передачи через радиоканал.
# В этом блоке только конфигурационные параметры.
# Классы с созвездиями и логикой modulate/demodulate — в Блоке 3.
# ================================================================


# ================================================================
# СПРАВОЧНИК МОДУЛЯЦИЙ
# ================================================================
#
# BITS_PER_SYMBOL (k) — число бит на один комплексный символ.
# M                   — размер созвездия (число точек), M = 2^k.
#
# BPSK  (k=1, M=2):  1 бит → {-1, +1}. Максимальная помехоустойчивость.
# QPSK  (k=2, M=4):  2 бита → 4 точки на окружности. Вдвое быстрее BPSK.
# QAM16 (k=4, M=16): 4 бита → сетка 4×4. Вчетверо быстрее BPSK.
# QAM64 (k=6, M=64): 6 бит  → сетка 8×8. Очень чувствительна к шуму.
# ================================================================

MODULATION_CONFIG = {
    "BPSK":  {"BITS_PER_SYMBOL": 1, "M": 2},
    "QPSK":  {"BITS_PER_SYMBOL": 2, "M": 4},
    "QAM16": {"BITS_PER_SYMBOL": 4, "M": 16},
    "QAM64": {"BITS_PER_SYMBOL": 6, "M": 64},
}

# Активная модуляция для отладки/ручного запуска одной модели.
# При полном эксперименте (Блок 8) перебираются все из MOD_TO_RUN.
MODULATION = "QPSK"

# Возможные варианты:
# MODULATION = "BPSK"
# MODULATION = "QPSK"
# MODULATION = "QAM16"
# MODULATION = "QAM64"

MOD_CFG         = MODULATION_CONFIG[MODULATION]
BITS_PER_SYMBOL = MOD_CFG["BITS_PER_SYMBOL"]   # k: бит на символ
M_ORDER         = MOD_CFG["M"]                  # M: размер созвездия

# Суммарное число бит, которые несёт один OFDM-символ.
# Именно столько бит предсказывает нейросеть на выходе.
# Пример: N_DATA=124, BITS_PER_SYMBOL=2 (QPSK) → N_BITS_PER_OFDM=248.
N_BITS_PER_OFDM = N_DATA * BITS_PER_SYMBOL

# Список модуляций для Блока 8 (полный эксперимент).
# Можно сократить для ускорения, например: ["BPSK", "QPSK"]
MOD_TO_RUN = ["BPSK", "QPSK", "QAM16", "QAM64"]

# Если True — рисуем теоретическую BER для AWGN (нижняя граница,
# без замираний). В реальном канале BER всегда выше теории.
ENABLE_THEORY_BER = True

# SNR для диаграмм созвездий (Блок 7): наглядно видно сжатие «облаков».
CONSTELLATION_SNR_LIST  = [0, 10, 20]
CONSTELLATION_N_SYMBOLS = 2000   # × N_DATA = итого точек на диаграмме



print("=" * 65)
print("ПАРАМЕТРЫ МОДУЛЯЦИИ")
print("=" * 65)
print(f"Активная модуляция: {MODULATION}  M={M_ORDER}  k={BITS_PER_SYMBOL}  N_BITS/OFDM={N_BITS_PER_OFDM}")
print(f"MOD_TO_RUN: {MOD_TO_RUN}")
print(f"USE_SNR_FEATURE={USE_SNR_FEATURE}  USE_INPUT_NORMALIZATION={USE_INPUT_NORMALIZATION}")
print("\n✓ Блок 2 успешно выполнен")

In [ ]:
# ================================================================
# БЛОК 2.5: GUI КОНФИГУРАТОР ЭКСПЕРИМЕНТОВ
# ================================================================
# Графический интерфейс для удобного изменения параметров
# и управления запуском экспериментов
# ================================================================

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

class ExperimentGUI:
    """GUI для управления параметрами экспериментов"""
    
    def __init__(self):
        self.config = {}
        self.create_widgets()
        
    def create_widgets(self):
        """Создаёт все виджеты интерфейса"""
        
        # ── Вкладка 1: Основные параметры ──
        self.research_mode = widgets.Dropdown(
            options=['END_TO_END', 'CHANNEL_ESTIMATION', 'SIGNAL_DETECTION'],
            value=RESEARCH_MODE,
            description='Режим:',
            style={'description_width': 'initial'}
        )
        
        self.channel_model = widgets.Dropdown(
            options=['ETU', 'EVA', 'EPA', 'RAYLEIGH', 'RICIAN'],
            value=CHANNEL_MODEL,
            description='Канал:',
            style={'description_width': 'initial'}
        )
        
        self.scenario = widgets.Dropdown(
            options=['BASELINE', 'CHALLENGING', 'DNN_FRIENDLY'],
            value=SCENARIO,
            description='Сценарий:',
            style={'description_width': 'initial'}
        )
        
        self.pa_nonlinearity = widgets.Checkbox(
            value=USE_PA_NONLINEARITY,
            description='Нелинейность PA',
            style={'description_width': 'initial'}
        )
        
        self.pa_ibo = widgets.FloatSlider(
            value=PA_IBO_DB,
            min=0.5,
            max=10.0,
            step=0.5,
            description='IBO (dB):',
            style={'description_width': 'initial'}
        )
        
        self.pa_rapp_p = widgets.FloatSlider(
            value=PA_RAPP_P,
            min=1.0,
            max=10.0,
            step=0.5,
            description='Rapp p:',
            style={'description_width': 'initial'}
        )
        
        tab1 = widgets.VBox([
            self.research_mode,
            self.channel_model,
            self.scenario,
            widgets.HBox([self.pa_nonlinearity, self.pa_ibo, self.pa_rapp_p])
        ])
        
        # ── Вкладка 2: Параметры обучения ──
        self.batch_size = widgets.IntSlider(
            value=BATCH_SIZE,
            min=256,
            max=8192,
            step=256,
            description='Batch size:',
            style={'description_width': 'initial'}
        )
        
        self.joint_epochs = widgets.IntSlider(
            value=JOINT_EPOCHS,
            min=20,
            max=300,
            step=10,
            description='Joint Epochs:',
            style={'description_width': 'initial'}
        )
        
        self.joint_transfer = widgets.Checkbox(
            value=JOINT_TRANSFER_WEIGHTS,
            description='Перенос весов ChanEstNet',
            style={'description_width': 'initial'}
        )
        
        self.use_snr_feature = widgets.Checkbox(
            value=USE_SNR_FEATURE,
            description='SNR как признак',
            style={'description_width': 'initial'}
        )
        
        self.use_input_norm = widgets.Checkbox(
            value=USE_INPUT_NORMALIZATION,
            description='Нормализация входа',
            style={'description_width': 'initial'}
        )
        
        tab2 = widgets.VBox([
            self.batch_size,
            self.joint_epochs,
            self.joint_transfer,
            widgets.HBox([self.use_snr_feature, self.use_input_norm])
        ])
        
        # ── Вкладка 3: Архитектура JointNet ──
        self.joint_arch = widgets.Dropdown(
            options=[
                ('Conv1D (локальная)', 'conv'),
                ('Dense (глобальная)', 'dense'),
            ],
            value='conv',
            description='Детектор:',
            style={'description_width': 'initial'}
        )
        
        self.det_dense_1 = widgets.IntText(
            value=512,
            description='Dense 1:',
            style={'description_width': 'initial'}
        )
        
        self.det_dense_2 = widgets.IntText(
            value=256,
            description='Dense 2:',
            style={'description_width': 'initial'}
        )
        
        tab3 = widgets.VBox([
            self.joint_arch,
            widgets.HBox([self.det_dense_1, self.det_dense_2])
        ])
        
        # ── Вкладка 4: Выбор вывода ──
        self.show_constellations = widgets.Checkbox(
            value=True,
            description='Созвездия',
            style={'description_width': 'initial'}
        )
        
        self.show_constellations_comparison = widgets.Checkbox(
            value=True,
            description='Сравнение созвездий',
            style={'description_width': 'initial'}
        )
        
        self.show_raw_signal = widgets.Checkbox(
            value=False,
            description='Сырой сигнал',
            style={'description_width': 'initial'}
        )
        
        self.show_benchmark = widgets.Checkbox(
            value=True,
            description='Бенчмарк производительности',
            style={'description_width': 'initial'}
        )
        
        self.show_ber_curves = widgets.Checkbox(
            value=True,
            description='BER кривые',
            style={'description_width': 'initial'}
        )
        
        self.show_ber_tables = widgets.Checkbox(
            value=True,
            description='BER таблицы',
            style={'description_width': 'initial'}
        )
        
        tab4 = widgets.GridBox([
            self.show_constellations,
            self.show_constellations_comparison,
            self.show_raw_signal,
            self.show_benchmark,
            self.show_ber_curves,
            self.show_ber_tables
        ], layout=widgets.Layout(grid_template_columns="repeat(2, 1fr)"))
        
        # ── Сборка табов ──
        tabs = widgets.Tab([tab1, tab2, tab3, tab4])
        tabs.set_title(0, 'Основные')
        tabs.set_title(1, 'Обучение')
        tabs.set_title(2, 'Архитектура')
        tabs.set_title(3, 'Вывод')
        
        # ── Кнопки управления ──
        self.apply_btn = widgets.Button(
            description='Применить параметры',
            button_style='primary',
            icon='check'
        )
        
        self.run_btn = widgets.Button(
            description='Запустить эксперимент',
            button_style='success',
            icon='play'
        )
        
        self.stop_btn = widgets.Button(
            description='Остановить',
            button_style='danger',
            icon='stop'
        )
        
        self.apply_btn.on_click(self.apply_config)
        self.run_btn.on_click(self.run_experiment)
        
        # ── Область вывода ──
        self.output = widgets.Output()
        
        # ── Финальная сборка ──
        self.gui = widgets.VBox([
            widgets.HTML("<h2>⚙️ Конфигуратор экспериментов</h2>"),
            tabs,
            widgets.HBox([self.apply_btn, self.run_btn, self.stop_btn]),
            self.output
        ])
        
        self.stop_requested = False
        self.stop_btn.on_click(lambda b: setattr(self, 'stop_requested', True))
        
    def apply_config(self, b=None):
        """Применяет параметры из GUI к глобальным переменным"""
        global RESEARCH_MODE, CHANNEL_MODEL, SCENARIO
        global USE_PA_NONLINEARITY, PA_IBO_DB, PA_RAPP_P
        global BATCH_SIZE, JOINT_EPOCHS, JOINT_TRANSFER_WEIGHTS
        global USE_SNR_FEATURE, USE_INPUT_NORMALIZATION, ARCH_CONFIG
        
        with self.output:
            clear_output()
            print("⚙️ Применяем параметры...")
            
            # Основные параметры
            RESEARCH_MODE = self.research_mode.value
            CHANNEL_MODEL = self.channel_model.value
            SCENARIO = self.scenario.value
            
            # Нелинейность
            USE_PA_NONLINEARITY = self.pa_nonlinearity.value
            PA_IBO_DB = self.pa_ibo.value
            PA_RAPP_P = self.pa_rapp_p.value
            
            # Обучение
            BATCH_SIZE = self.batch_size.value
            JOINT_EPOCHS = self.joint_epochs.value
            JOINT_TRANSFER_WEIGHTS = self.joint_transfer.value
            USE_SNR_FEATURE = self.use_snr_feature.value
            USE_INPUT_NORMALIZATION = self.use_input_norm.value
            
            # Архитектура
            if self.joint_arch.value == 'dense':
                ARCH_CONFIG['JointNet']['det_dense_layers'] = [
                    self.det_dense_1.value,
                    self.det_dense_2.value
                ]
                # Удаляем conv_layers если есть
                if 'det_conv_layers' in ARCH_CONFIG['JointNet']:
                    del ARCH_CONFIG['JointNet']['det_conv_layers']
            else:
                ARCH_CONFIG['JointNet']['det_conv_layers'] = [
                    (64, 1), (128, 3), (128, 3), (128, 3), (64, 1)
                ]
                if 'det_dense_layers' in ARCH_CONFIG['JointNet']:
                    del ARCH_CONFIG['JointNet']['det_dense_layers']
            
            # Пересоздаём имя папки
            global EXP_DIR_NAME, BASE_SAVE_DIR
            EXP_DIR_NAME = generate_experiment_dir_name()
            BASE_SAVE_DIR = EXP_DIR_NAME
            os.makedirs(BASE_SAVE_DIR, exist_ok=True)
            
            print(f"✅ Параметры применены!")
            print(f"📂 Папка: {BASE_SAVE_DIR}/")
            
    def run_experiment(self, b=None):
        """Запускает эксперимент с текущими параметрами"""
        self.stop_requested = False
        
        with self.output:
            clear_output()
            print("🚀 Запуск эксперимента...")
            print(f"📂 Папка: {BASE_SAVE_DIR}/")
            
            try:
                # Получаем настройки вывода
                outputs = {
                    'constellations': self.show_constellations.value,
                    'constellations_comparison': self.show_constellations_comparison.value,
                    'raw_signal': self.show_raw_signal.value,
                    'benchmark': self.show_benchmark.value,
                    'ber_curves': self.show_ber_curves.value,
                    'ber_tables': self.show_ber_tables.value
                }
                
                # Запускаем эксперимент
                run_experiment_with_gui(outputs, self)
                
            except Exception as e:
                print(f"❌ Ошибка: {e}")
                import traceback
                traceback.print_exc()
                
    def display(self):
        """Отображает GUI"""
        display(self.gui)

# Создаём экземпляр GUI
experiment_gui = ExperimentGUI()

# Отображаем GUI
experiment_gui.display()

In [ ]:
# ================================================================
# БЛОК 3
# МОДУЛЯЦИИ, МОДЕЛЬ КАНАЛА И OFDM-ГЕНЕРАТОР ВЫБОРОК
# ================================================================
#
# 3.1  Классы модуляций с ML-детекторами и теоретическими BER
# 3.2  3GPP-профили задержек-мощностей (PDP) и генератор отводов канала
# 3.3  DFT-based оценщик канала (шумоподавляющий)
# 3.4  Главный OFDM-генератор generate() с поддержкой PA-нелинейности
# 3.5  build_xy() / get_io_sizes() — подготовка данных для нейросети
# ================================================================

from scipy.special import erfc   # для теоретических BER-формул


# ================================================================
# 3.1 БАЗОВЫЙ КЛАСС МОДУЛЯЦИИ
# ================================================================
#
# Все конкретные модуляции наследуют от _BaseModulation три метода.
# Каждый подкласс задаёт только атрибуты: constellation, gray_table, M, k.
#
# КОД ГРЕЯ: соседние точки созвездия отличаются ровно одним битом.
# Это минимизирует BER: ошибка детектирования (переход в соседнюю
# точку из-за шума) в большинстве случаев даёт ошибку лишь в одном бите.
# Формула: gray(n) = n XOR (n >> 1).
# ================================================================

class _BaseModulation:
    """Базовый класс. Использовать только через подклассы."""

    def modulate(self, bits):
        """
        Кодирует биты в символы созвездия методом минимума расстояния Хэмминга.

        Для каждого входного слова из k битов ищется точка созвездия,
        чей код Грея наиболее близок к входному слову.

        bits    : (N, k) int — N символов по k бит каждый.
        Возвращает (N,) complex.
        """
        # hamming[i,j] = число различий между i-м словом и j-й строкой gray_table
        hamming = np.sum(
            bits[:, np.newaxis, :] != self.gray_table[np.newaxis, :, :],
            axis=2
        )
        return self.constellation[np.argmin(hamming, axis=1)]

    def demodulate(self, symbols):
        """
        ML-детектор: выбирает ближайшую точку созвездия (евклидово расстояние).
        Оптимален при AWGN и равновероятных символах.

        symbols : (N,) complex — принятые (уже эквализованные) символы.
        Возвращает (N, k) int — предсказанные биты.
        """
        dists = np.abs(symbols[:, np.newaxis] - self.constellation[np.newaxis, :])
        return self.gray_table[np.argmin(dists, axis=1)]

    def ber_theory(self, snr_lin):
        """
        Теоретическая BER для квадратного M-QAM в AWGN-канале (аппроксимация):
            BER ≈ 4*(1 - 1/√M) / k * Q(√(3*Es/N0/(M-1)))
        где Q(x) = 0.5*erfc(x/√2), snr_lin = Es/N0 (линейный).
        Точна для BPSK/QPSK, незначительно завышает для QAM16/64.
        """
        coeff = 4.0 * (1.0 - 1.0 / np.sqrt(self.M)) / self.k
        arg   = 3.0 * snr_lin / (2.0 * (self.M - 1))
        return coeff * 0.5 * erfc(np.sqrt(arg))


class BPSK(_BaseModulation):
    """
    Binary Phase Shift Keying. 2 точки на вещественной оси.
    Бит 0 → -1+0j,  бит 1 → +1+0j.
    Максимальная помехоустойчивость: расстояние между точками = 2.
    """
    name = 'BPSK'; M = 2; k = 1
    constellation = np.array([-1.0+0j, +1.0+0j])
    gray_table    = np.array([[0], [1]], dtype=np.int32)

    def ber_theory(self, snr_lin):
        # Точная формула для BPSK: BER = Q(√(2*Eb/N0)) = 0.5*erfc(√(Es/N0))
        return 0.5 * erfc(np.sqrt(snr_lin))


class QPSK(_BaseModulation):
    """
    Quadrature Phase Shift Keying. 4 точки на единичной окружности.
    Коэффициент 1/√2 нормирует среднюю мощность символа: E[|X|²]=1.
    Расположение (код Грея — соседи отличаются 1 битом):
        00 → III квадрант,  01 → II,  10 → IV,  11 → I.
    """
    name = 'QPSK'; M = 4; k = 2
    _s = 1.0 / np.sqrt(2)
    constellation = np.array([
        -_s - _s*1j,   # 00
        -_s + _s*1j,   # 01
        +_s - _s*1j,   # 10
        +_s + _s*1j,   # 11
    ])
    gray_table = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=np.int32)
    # ber_theory наследуется из _BaseModulation (точна для QPSK)


def _make_square_qam(M):
    """
    Строит квадратное M-QAM созвездие с кодированием Грея.

    Алгоритм:
      1. Сетка sqM×sqM точек, координаты: {-(sqM-1), ..., -1, +1, ..., sqM-1}.
      2. Для каждой точки вычисляем код Грея: gray(n) = n XOR (n>>1).
         Первые k/2 битов → I-координата (Re), вторые k/2 → Q (Im).
      3. Нормировка: делим на √(E[|pts|²]) → E[|X|²]=1.
    M должно быть степенью 4: 16, 64, 256, ...
    """
    sqM  = int(np.sqrt(M))
    k    = int(np.log2(M))
    half = k // 2

    def gray1d(n): return n ^ (n >> 1)

    axis = np.arange(-(sqM-1), sqM, 2)
    pts, bits_list = [], []
    for qi, yv in enumerate(axis[::-1]):    # Q-ось: сверху вниз
        for ii, xv in enumerate(axis):      # I-ось: слева направо
            pts.append(xv + 1j*yv)
            gi = gray1d(ii); gq = gray1d(qi)
            b = []
            for p in range(half-1, -1, -1): b.append((gi >> p) & 1)  # биты I
            for p in range(half-1, -1, -1): b.append((gq >> p) & 1)  # биты Q
            bits_list.append(b)
    pts   = np.array(pts, dtype=complex)
    table = np.array(bits_list, dtype=np.int32)
    pts  /= np.sqrt(np.mean(np.abs(pts)**2))   # E[|X|²] = 1
    return pts, table


class QAM16(_BaseModulation):
    """16-QAM. 4 бита/символ, 16 точек на сетке 4×4."""
    name = 'QAM16'; M = 16; k = 4
    constellation, gray_table = _make_square_qam(16)


class QAM64(_BaseModulation):
    """64-QAM. 6 бит/символ, 64 точки на сетке 8×8."""
    name = 'QAM64'; M = 64; k = 6
    constellation, gray_table = _make_square_qam(64)


# Реестр модуляций строится из MODULATION_CONFIG.
# Если добавить модуляцию в конфиг без класса — assert самопроверки укажет на ошибку.
_MODULATION_CLASSES = {'BPSK': BPSK, 'QPSK': QPSK, 'QAM16': QAM16, 'QAM64': QAM64}
MODULATIONS = {name: cls()
               for name, cls in _MODULATION_CLASSES.items()
               if name in MODULATION_CONFIG}


# ================================================================
# 3.2 МОДЕЛЬ КАНАЛА: ПРОФИЛИ МОЩНОСТИ ОТВОДОВ (PDP)
# ================================================================
#
# _PDP_DB — Power Delay Profile в дБ из 3GPP TS 36.101, Annex B.
#
# УПРОЩЕНИЕ: реальные профили задают задержки в наносекундах.
# Здесь i-й отвод расположен на задержке i*T_s (1 отвод = 1 период
# дискретизации). Стандартное академическое приближение: даёт правильную
# относительную частотную избирательность (ETU заметно избирательнее EPA),
# хотя точная форма АЧХ отличается от стандарта.
# ================================================================

_PDP_DB = {
    # Мощности в дБ для каждого отвода (нормируются в _tap_power_profile)
    'EPA': [ 0.0, -1.0, -2.0,  -3.0,  -8.0, -17.2, -20.8],          # 7 отводов
    'EVA': [ 0.0, -1.5, -1.4,  -3.6,  -0.6,  -9.1,  -7.0, -12.0, -16.9],  # 9
    'ETU': [-1.0, -1.0, -1.0,   0.0,   0.0,   0.0,  -3.0,  -5.0,  -7.0],  # 9
}


def _tap_power_profile(channel_model, n_taps):
    """
    Возвращает (n_taps,) нормированных мощностей отводов, sum=1.

    Rayleigh/Rician: равномерный профиль p[i] = 1/n_taps.
    EPA/EVA/ETU: из таблицы _PDP_DB, конвертируем дБ → линейный,
    нормируем. Если n_taps > длины таблицы — дополняем слабейшим отводом.
    """
    if channel_model in ('RAYLEIGH', 'RICIAN'):
        p = np.ones(n_taps)
    elif channel_model in _PDP_DB:
        p_db = np.array(_PDP_DB[channel_model][:n_taps], dtype=float)
        if len(p_db) < n_taps:
            pad  = np.full(n_taps - len(p_db), p_db.min())
            p_db = np.concatenate([p_db, pad])
        p = 10.0 ** (p_db / 10.0)   # дБ → линейный
    else:
        raise ValueError(f"Неизвестная модель канала: {channel_model}")
    return p / p.sum()   # нормировка: суммарная мощность = 1


# Профиль вычисляется один раз при запуске блока
TAP_POWER_PROFILE = _tap_power_profile(CHANNEL_MODEL, N_TAPS)


def generate_channel_taps(N):
    """
    Генерирует N реализаций импульсной характеристики канала.

    Возвращает h формы (N, N_TAPS) complex. Гарантия: E[sum|h|²]=1.

    Rayleigh: h[:,n] ~ CN(0, p[n]) — комплексная гауссова СВ с нулевым
    средним и дисперсией p[n]=TAP_POWER_PROFILE[n].
    Реализация: Re ~ N(0, √(p/2)),  Im ~ N(0, √(p/2)).

    Rician: нулевой отвод получает детерминированную LOS-компоненту
    мощностью K/(K+1). NLOS масштабируется на 1/(K+1) → суммарная
    мощность остаётся = 1.
    """
    p = TAP_POWER_PROFILE
    if CHANNEL_MODEL == 'RICIAN':
        K_lin      = 10.0 ** (K_FACTOR_DB / 10.0)
        nlos_power = p / (K_lin + 1.0)
        h = ((np.random.randn(N, N_TAPS) + 1j*np.random.randn(N, N_TAPS))
             * np.sqrt(nlos_power / 2.0))
        # Детерминированный LOS-вектор в нулевом отводе (вещественная ось)
        h[:, 0] += np.sqrt(K_lin / (K_lin + 1.0))
    else:
        h = ((np.random.randn(N, N_TAPS) + 1j*np.random.randn(N, N_TAPS))
             * np.sqrt(p / 2.0))
    return h


# ================================================================
# 3.3 DFT-BASED ОЦЕНЩИК КАНАЛА
# ================================================================
#
# Алгоритм LS + DFT-based шумоподавление:
#
#   Шаг 1. X_pilot=1+0j → Y_pilot = H_pilot + шум (прямая LS-оценка).
#   Шаг 2. IFFT длиной N_PILOT от Y_pilot → приближение к h (отводы).
#           При равномерных пилотах и N_TAPS≤N_PILOT — без алиасинга.
#   Шаг 3. Обнуляем отсчёты с индексами ≥ N_TAPS.
#           Шум равномерно размазан по всем N_PILOT отсчётам,
#           канал сосредоточен в первых N_TAPS → зануление убирает шум
#           «невозможных» задержек без потери информации о канале.
#   Шаг 4. FFT длиной N_FFT → гладкая оценка H на ВСЕХ N_FFT поднесущих.
#
# Если N_TAPS >= N_PILOT: алиасинг на шаге 2, шумоподавление не работает,
# но оценщик не падает. Об этом выводится предупреждение.
# ================================================================

_N_EFF_TAPS = min(N_TAPS, N_PILOT)   # отводов, которые можно разрешить

if N_TAPS >= N_PILOT:
    print(f"[Блок 3] ⚠ N_TAPS={N_TAPS} >= N_PILOT={N_PILOT}: "
          f"DFT-оценщик не подавляет шум (алиасинг отводов).")


def estimate_channel(Y_pilot):
    """
    DFT-based LS-оценщик с шумоподавлением.

    Y_pilot : (N, N_PILOT) complex — принятые пилоты (X_pilot=1 → Y=H+шум).
    Возвращает (H_est_data, H_est_pilot): (N, N_DATA) и (N, N_PILOT).
    """
    h_alias = np.fft.ifft(Y_pilot, n=N_PILOT, axis=1)   # шаг 2: IFFT
    h_trunc = np.zeros_like(h_alias)
    h_trunc[:, :_N_EFF_TAPS] = h_alias[:, :_N_EFF_TAPS] # шаг 3: обнуление
    H_full  = np.fft.fft(h_trunc, n=N_FFT, axis=1)      # шаг 4: FFT → все поднесущие
    return H_full[:, DATA_IDX], H_full[:, PILOT_IDX]


# ================================================================
# 3.4 ГЛАВНЫЙ OFDM-ГЕНЕРАТОР
# ================================================================
#
# generate() реализует полную цепочку за один вызов:
#   биты → модуляция → OFDM-символ → [PA] → канал → шум → оценка H
#
# КЛЮЧЕВОЕ СВОЙСТВО OFDM: в частотной области канал диагональный:
#   Y[k] = H[k]*X[k] + N[k]  (поэлементное умножение, без ISI!)
# Это следствие двух фактов:
#   а) CP превращает линейную свёртку в циклическую;
#   б) FFT диагонализирует любую циrculant-матрицу.
#
# Возвращаемый словарь содержит все промежуточные величины,
# которые нужны детекторам (Блок 6) и build_xy() (Блок 3.5).
# ================================================================

def generate(N_samples, SNR_dB, mod_name):
    """
    Полная OFDM-симуляция: биты → принятый сигнал.

    N_samples : int   — число OFDM-символов (одна выборка = один символ).
    SNR_dB    : float — отношение сигнал/шум в дБ.
    mod_name  : str   — имя модуляции из MODULATIONS.

    Возвращает словарь:
        bits         : (N, N_DATA*k) int32  — переданные биты
        snr_dB       : float                — уровень SNR
        Y_freq       : (N, N_FFT) complex   — принятый OFDM-символ (все поднесущие)
        Y_data       : (N, N_DATA) complex  — только data-поднесущие
        Y_pilot      : (N, N_PILOT) complex — только pilot-поднесущие
        H_true_data  : (N, N_DATA) complex  — истинный H на data (для Oracle)
        H_true_pilot : (N, N_PILOT) complex — истинный H на pilot
        H_est_data   : (N, N_DATA) complex  — DFT-оценка H на data
        H_est_pilot  : (N, N_PILOT) complex — DFT-оценка H на pilot
        eq_symbols   : (N, N_DATA) complex  — ZF-эквализованные символы
        norm         : (N, 1) float32       — RMS пилотов (для нормировки входа)
        X_pa_freq    : (N, N_FFT) complex   — сигнал после PA (до канала)
    """
    mod = MODULATIONS[mod_name]
    k   = mod.k
    N   = N_samples

    # ─ Шаг 1: случайные биты и модуляция ────────────────────────
    bits      = np.random.randint(0, 2, (N * N_DATA, k), dtype=np.int32)
    data_syms = mod.modulate(bits).reshape(N, N_DATA)

    # ─ Шаг 2: формирование OFDM-символа ──────────────────────────
    # Пилоты = 1+0j (известны обеим сторонам), данные — символы
    X_freq = np.zeros((N, N_FFT), dtype=complex)
    X_freq[:, PILOT_IDX] = 1.0 + 0j
    X_freq[:, DATA_IDX]  = data_syms

    # ─ Шаг 3 (опционально): нелинейный усилитель мощности ────────
    # Модель Rapp применяется во временной области (усилитель работает
    # с аналоговым сигналом — после IFFT, до канала).
    if USE_PA_NONLINEARITY:
        X_time = np.fft.ifft(X_freq, n=N_FFT, axis=1)   # → временная обл.
        amp    = np.abs(X_time)
        # numpy.ifft делит на N_FFT → RMS = 1/√N_FFT → масштабируем A_sat
        A_sat  = (1.0 / np.sqrt(N_FFT)) * 10.0 ** (PA_IBO_DB / 20.0) #!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!возможно ошибка - перед РА
        p_rapp = PA_RAPP_P
        # AM/AM: ограничение амплитуды по модели Rapp (фазу не трогаем)
        amp_out   = amp / ((1.0 + (amp/A_sat)**(2*p_rapp))**(1.0/(2*p_rapp)))
        X_time_pa = amp_out * np.exp(1j * np.angle(X_time))
        X_pa_freq = np.fft.fft(X_time_pa, n=N_FFT, axis=1)  # → частотная обл.
    else:
        X_pa_freq = X_freq   # линейный усилитель

    # ─ Шаг 4: случайный канал ────────────────────────────────────
    h      = generate_channel_taps(N)          # (N, N_TAPS) временная обл.
    H_freq = np.fft.fft(h, n=N_FFT, axis=1)   # (N, N_FFT) частотная обл.

    # ─ Шаг 5: AWGN-шум ───────────────────────────────────────────
    # При нормированных мощностях сигнала и канала: SNR = 1/(2*noise_std²)
    # → noise_std = √(0.5 / SNR_lin)
    SNR_lin   = 10.0 ** (SNR_dB / 10.0)
    noise_std = np.sqrt(0.5 / SNR_lin)
    noise     = noise_std * (np.random.randn(N, N_FFT) + 1j*np.random.randn(N, N_FFT))

    # ─ Шаг 6: Y[k] = H[k]*X[k] + N[k] — поэлементное умножение! ─
    Y_freq = X_pa_freq * H_freq + noise

    # ─ Шаг 7: разделение на pilot и data ─────────────────────────
    Y_data        = Y_freq[:, DATA_IDX]
    Y_pilot       = Y_freq[:, PILOT_IDX]
    H_true_data   = H_freq[:, DATA_IDX]
    H_true_pilot  = H_freq[:, PILOT_IDX]

    # ─ Шаг 8: DFT-оценка канала ──────────────────────────────────
    H_est_data, H_est_pilot = estimate_channel(Y_pilot)

    # ─ Шаг 9: ZF-эквализация (для созвездий и SIGNAL_DETECTION) ──
    # Защита от деления на ноль при глубоком замирании (|H| → 0)
    H_safe     = np.where(np.abs(H_est_data) < 1e-9, 1e-9+0j, H_est_data)
    eq_symbols = Y_data / H_safe   # |eq_symbols| ≈ |X| при хорошей оценке H

    # ─ Шаг 10: нормировочный коэффициент по RMS пилотов ──────────
    # norm ≈ √SNR_lin при E[|H|²]=1. Используется в build_xy()
    # для нормировки входа нейросети (USE_INPUT_NORMALIZATION).
    norm = np.sqrt(np.mean(np.abs(Y_pilot)**2, axis=1, keepdims=True))
    norm = np.maximum(norm, 1e-9)   # защита от нуля

    return {
        'bits'        : bits.reshape(N, N_DATA * k).astype(np.int32),
        'snr_dB'      : SNR_dB,
        'Y_freq'      : Y_freq,
        'Y_data'      : Y_data,
        'Y_pilot'     : Y_pilot,
        'H_true_data' : H_true_data,
        'H_true_pilot': H_true_pilot,
        'H_est_data'  : H_est_data,
        'H_est_pilot' : H_est_pilot,
        'eq_symbols'  : eq_symbols,
        'norm'        : norm,
        'X_pa_freq'   : X_pa_freq,
    }


# ================================================================
# 3.5 ПОДГОТОВКА ВХОДА/ВЫХОДА НЕЙРОСЕТИ
# ================================================================
#
# build_xy() преобразует словарь generate() в (X, y) для нейросети.
# Формат зависит от RESEARCH_MODE:
#
# CHANNEL_ESTIMATION:
#   X = [Re(Y_pilot_norm), Im(Y_pilot_norm), (snr)]  — 2*N_PILOT + (1)
#   y = [Re(H_data_norm),  Im(H_data_norm)]           — 2*N_DATA
#   Вход и цель нормируются на одно norm. При денормализации H_pred:
#   умножить на norm, чтобы восстановить абсолютный масштаб.
#
# SIGNAL_DETECTION:
#   Z = eq_symbols = Y_data/H_est (ZF, |Z|≈1, уже «нормирован»)
#   X = [Re(Z), Im(Z), (snr)]  — 2*N_DATA + (1)
#   y = bits. Дополнительная нормировка по пилотам не нужна.
#
# END_TO_END:
#   X = [Re(Y_freq), Im(Y_freq), (snr)]  — 2*N_FFT + (1)
#   y = bits. Вход — ВЕСЬ OFDM-символ, пилоты на своих позициях.
#   Это важно для ConvNet: частотная структура сигнала не нарушена.
# ================================================================

def _snr_feature(N, snr_dB):
    """
    Нормирует SNR в [0,1] относительно [SNR_TRAIN_MIN, SNR_TRAIN_MAX].
    Возвращает столбец (N, 1) float32.
    Позволяет одной модели адаптировать поведение к текущему уровню шума.
    """
    span = float(SNR_TRAIN_MAX - SNR_TRAIN_MIN)
    val  = (snr_dB - SNR_TRAIN_MIN) / span
    return np.full((N, 1), val, dtype=np.float32)


def build_xy(g, research_mode=None):
    """
    Преобразует словарь generate() в (X, y) float32 для нейросети.

    g             : dict — результат generate().
    research_mode : str или None (None → берётся глобальный RESEARCH_MODE).
    """
    if research_mode is None:
        research_mode = RESEARCH_MODE
    N = g['bits'].shape[0]

    if research_mode == 'CHANNEL_ESTIMATION':
        Yp = g['Y_pilot']
        Ht = g['H_true_data']
        if USE_INPUT_NORMALIZATION:
            # Нормируем вход и цель одновременно. При использовании
            # H_pred нужна денормализация: H_pred_abs = H_pred_norm * norm.
            Yp = Yp / g['norm']
            Ht = Ht / g['norm']
        X = np.hstack([np.real(Yp), np.imag(Yp)])
        y = np.hstack([np.real(Ht), np.imag(Ht)]).astype(np.float32)

    elif research_mode == 'SIGNAL_DETECTION':
        # eq_symbols = Y/H_est — уже в масштабе созвездия (|Z|≈1)
        Z = g['eq_symbols']
        X = np.hstack([np.real(Z), np.imag(Z)])
        y = g['bits'].astype(np.float32)

    elif research_mode == 'END_TO_END':
        Yf = g['Y_freq']
        if USE_INPUT_NORMALIZATION:
            Yf = Yf / g['norm']
        X = np.hstack([np.real(Yf), np.imag(Yf)])
        y = g['bits'].astype(np.float32)

    else:
        raise ValueError(f"Неизвестный RESEARCH_MODE: {research_mode}")

    if USE_SNR_FEATURE:
        X = np.hstack([X, _snr_feature(N, g['snr_dB'])])

    return X.astype(np.float32), y


def get_io_sizes(mod_name, research_mode=None):
    """
    Возвращает (input_size, output_size, n_freq, loss, final_activation).

    n_freq — длина «частотной оси» для ConvNet:
        CHANNEL_ESTIMATION → N_PILOT  (свёртка по пилотам)
        SIGNAL_DETECTION   → N_DATA   (свёртка по data-поднесущим)
        END_TO_END         → N_FFT    (свёртка по всем поднесущим)
    """
    if research_mode is None:
        research_mode = RESEARCH_MODE
    mod   = MODULATIONS[mod_name]
    extra = 1 if USE_SNR_FEATURE else 0

    if research_mode == 'CHANNEL_ESTIMATION':
        n_freq = N_PILOT
        return 2*n_freq + extra, 2*N_DATA, n_freq, 'mse', 'linear'
    elif research_mode == 'SIGNAL_DETECTION':
        n_freq = N_DATA
        return 2*n_freq + extra, N_DATA*mod.k, n_freq, 'binary_crossentropy', 'sigmoid'
    elif research_mode == 'END_TO_END':
        n_freq = N_FFT
        return 2*n_freq + extra, N_DATA*mod.k, n_freq, 'binary_crossentropy', 'sigmoid'
    else:
        raise ValueError(f"Неизвестный RESEARCH_MODE: {research_mode}")


# ── Самопроверка Блока 3 ─────────────────────────────────────────
print("=" * 65)
print("БЛОК 3: МОДУЛЯЦИИ, КАНАЛ, ГЕНЕРАТОР ВЫБОРОК")
print("=" * 65)
assert set(MODULATIONS) == set(MODULATION_CONFIG)
for name, mod in MODULATIONS.items():
    cfg = MODULATION_CONFIG[name]
    assert mod.M == cfg['M'] and mod.k == cfg['BITS_PER_SYMBOL'], name
print(f"Канал: {CHANNEL_MODEL}  (N_TAPS={N_TAPS})")
print(f"  Профиль мощности : {np.round(TAP_POWER_PROFILE, 4)}  (сумма={TAP_POWER_PROFILE.sum():.4f})")
in_sz, out_sz, n_freq, loss, act = get_io_sizes(MODULATION)
print(f"Режим {RESEARCH_MODE}: вход={in_sz}  выход={out_sz}  loss={loss}  n_freq={n_freq}")
print("Тест modulate/demodulate и generate():")
for name in MODULATIONS:
    syms      = MODULATIONS[name].modulate(MODULATIONS[name].gray_table)
    recovered = MODULATIONS[name].demodulate(syms)
    assert np.all(recovered == MODULATIONS[name].gray_table), name
    g_test = generate(5, 10.0, name)
    X_test, y_test = build_xy(g_test)
    in_sz_m, out_sz_m, *_ = get_io_sizes(name)
    assert X_test.shape == (5, in_sz_m) and y_test.shape == (5, out_sz_m)
    print(f"  {name:>6}: ✓  X{X_test.shape} y{y_test.shape}")
g_check = generate(20000, 10.0, MODULATION)
print(f"Нормировка канала: mean|H|²={np.mean(np.abs(g_check['H_true_data'])**2):.4f} (ожидание≈1)")
print("\n✓ Блок 3 успешно выполнен")

In [ ]:
# ================================================================
# БЛОК 4: АРХИТЕКТУРЫ НЕЙРОСЕТЕЙ
# ================================================================
#
# Специализированные архитектуры (трёхэтапный протокол):
#   A. ChanEstNet  — оценщик канала (Conv1D по пилотам + Dense)
#   B. DetectorNet — детектор символов (Conv1D по поднесущим)
#   C. JointNet    — совместная оценка + детектирование
#
# Классические архитектуры (для сравнения):
#   D. MLP       — полносвязная, базовый эталон
#   E. ResidualNet — с остаточными связями
#   F. ConvNet   — свёрточная по частотной оси
#
# Ключевые архитектурные решения:
#
#   BatchNormalization везде: нормировка активаций → ускорение
#   сходимости, возможность использовать более высокий LR.
#
#   ChanEstNet и JointNet имеют ОДИНАКОВЫЕ имена слоёв оценочной
#   ветви (conv1/2/3, bn_c1/2/3, fc1/2, bn_f1/2, h_pred).
#   train_joint() копирует предобученные веса по именам слоёв —
#   это критично для корректной передачи знаний между шагами 1 и 3.
#
#   DetectorNet использует Conv1D с kernel=1 («поэлементная свёртка»):
#   одно правило детектора для всех N_DATA поднесущих → быстрая
#   сходимость, физически обоснована (поднесущие однотипны).
#   Kernel=3 в средних слоях добавляет контекст соседей (частотная
#   корреляция канала помогает в переходных зонах созвездия).
# ================================================================


class _Slice(KL.Layer):
    """
    Keras-слой: вырезает x[:, start : start+length].

    Нужен потому, что стандартный Python-срез нельзя использовать
    внутри Keras функциональной модели — граф не может его отследить.
    get_config() обязателен для сохранения/загрузки модели:
        keras.models.load_model(path, custom_objects={'_Slice': _Slice})
    """
    def __init__(self, start, length, **kw):
        super().__init__(**kw)
        self.start  = int(start)
        self.length = int(length)

    def call(self, x):
        return x[:, self.start : self.start + self.length]

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'start': self.start, 'length': self.length})
        return cfg


# ================================================================
# A: ОЦЕНЩИК КАНАЛА (ChanEstNet)
# ================================================================
#
# Задача: по N_PILOT наблюдениям предсказать H на N_DATA поднесущих.
# Вход : [Re(Y_pilot_norm)*N_PILOT, Im(Y_pilot_norm)*N_PILOT, snr(1)]
# Выход: [Re(H_data_norm)*N_DATA,   Im(H_data_norm)*N_DATA]
# Loss : MSE
#
# Conv1D (kernel=3) — учитывает корреляцию 3 соседних пилотов
# («плавность» H по частоте). Dense — глобальный контекст для
# интерполяции на весь N_FFT (видит все пилоты сразу).
#
# КРИТИЧНО: имена conv1/2/3, bn_c1/2/3, fc1/2, bn_f1/2, h_pred
# должны совпадать с оценочной ветвью JointNet. Это позволяет
# train_joint() копировать веса по именам слоёв.
# ================================================================

def build_chan_est_net(mod_name):
    """Строит ChanEstNet с архитектурой из ARCH_CONFIG."""
    cfg = ARCH_CONFIG['ChanEstNet']
    extra  = 1 if USE_SNR_FEATURE else 0
    in_sz  = 2 * N_PILOT + extra
    out_sz = 2 * N_DATA
    inp = keras.Input(shape=(in_sz,), name='chan_est_input')

    re_p = _Slice(0,       N_PILOT, name='re_pilot')(inp)
    im_p = _Slice(N_PILOT, N_PILOT, name='im_pilot')(inp)
    re3  = KL.Reshape((N_PILOT, 1))(re_p)
    im3  = KL.Reshape((N_PILOT, 1))(im_p)
    grid = KL.Concatenate(axis=-1, name='pilot_grid')([re3, im3])

    # ── Динамические Conv1D-блоки ──────────────────────────────
    x = grid
    for i, (filters, kernel) in enumerate(cfg['conv_layers'], 1):
        x = KL.Conv1D(filters, kernel, padding='same', name=f'conv{i}')(x)
        x = KL.BatchNormalization(name=f'bn_c{i}')(x)
        x = KL.ReLU(name=f'relu_c{i}')(x)

    x = KL.Flatten()(x)

    if extra > 0:
        snr_f = _Slice(2 * N_PILOT, extra, name='snr_feat')(inp)
        x = KL.Concatenate()([x, snr_f])

    # ── Динамические Dense-блоки ───────────────────────────────
    for i, units in enumerate(cfg['dense_layers'], 1):
        x = KL.Dense(units, name=f'fc{i}')(x)
        x = KL.BatchNormalization(name=f'bn_f{i}')(x)
        x = KL.ReLU(name=f'relu_f{i}')(x)

    out = KL.Dense(out_sz, activation='linear', name='h_pred')(x)

    model = keras.Model(inp, out, name='ChanEstNet')
    n_params = model.count_params()
    n_conv = len(cfg['conv_layers'])
    n_dense = len(cfg['dense_layers'])
    print(f"  {'ChanEstNet':>12} | {mod_name:>8} | CHANNEL_ESTIMATION    | "
          f"вход={in_sz:>4} выход={out_sz:>4} loss=mse "
          f"conv={n_conv} dense={n_dense} парам={n_params:,}")
    return model, in_sz, out_sz


# ================================================================
# B: ДЕТЕКТОР СИМВОЛОВ (DetectorNet)
# ================================================================
#
# Задача: по эквализованному сигналу предсказать переданные биты.
# Вход: [Re(Z_clip)*N_DATA, Im(Z_clip)*N_DATA, Re(H_norm)*N_DATA, Im(H_norm)*N_DATA, snr]
#       размерность = 4*N_DATA + 1
#
# Z = Y_data / H_dnn — ZF-сигнал, |Z|≈1 всегда (не зависит от |H|).
# Z_clip: ограничение _Z_CLIP=3.0 убирает выбросы при замираниях.
# H_norm = H_dnn/norm: показывает качество каждой поднесущей
#   (|H_norm|≪1 → замирание → ZF усилил шум → не доверять данным).
#
# Выход:
#   k≤2 (BPSK/QPSK): N_DATA*k бит, activation=sigmoid, loss=BCE.
#   k≥4 (QAM16/64):  2*N_DATA символы Re/Im в ПЕРЕМЕЖЁННОМ порядке,
#                     activation=linear, loss=MSE.
#   Причина разделения: для QAM64 BCE по 6*N_DATA битам плохо
#   сходится — MSE по символу физически правильнее и быстрее.
#
# ВАЖНО О ПЕРЕМЕЖЁННОМ ПОРЯДКЕ:
#   Conv1D(out_per_sub, kernel=1) + Reshape даёт порядок:
#   [Re_s0, Im_s0, Re_s1, Im_s1, ..., Re_s(N-1), Im_s(N-1)]
#   Обучающие цели ДОЛЖНЫ иметь тот же порядок (см. _build_training_data_detector).
# ================================================================

def build_detector_net(mod_name):
    """Строит DetectorNet с архитектурой из ARCH_CONFIG."""
    cfg = ARCH_CONFIG['DetectorNet']
    mod    = MODULATIONS[mod_name]
    extra  = 1 if USE_SNR_FEATURE else 0
    in_sz  = 4 * N_DATA + extra
    out_sz    = N_DATA * mod.k if mod.k <= 2 else 2 * N_DATA
    final_act = 'sigmoid' if mod.k <= 2 else 'linear'
    inp = keras.Input(shape=(in_sz,), name='det_input')

    re_z = _Slice(0,          N_DATA, name='re_z')(inp)
    im_z = _Slice(N_DATA,     N_DATA, name='im_z')(inp)
    re_H = _Slice(2 * N_DATA, N_DATA, name='re_H')(inp)
    im_H = _Slice(3 * N_DATA, N_DATA, name='im_H')(inp)

    grid = KL.Concatenate(axis=-1, name='det_grid')([
        KL.Reshape((N_DATA, 1))(re_z),
        KL.Reshape((N_DATA, 1))(im_z),
        KL.Reshape((N_DATA, 1))(re_H),
        KL.Reshape((N_DATA, 1))(im_H),
    ])

    # ── Динамические Conv1D-блоки ──────────────────────────────
    x = grid
    for i, (filters, kernel) in enumerate(cfg['conv_layers'], 1):
        x = KL.Conv1D(filters, kernel, padding='same', name=f'd_conv{i}')(x)
        x = KL.BatchNormalization(name=f'd_bn{i}')(x)
        x = KL.ReLU(name=f'd_relu{i}')(x)

    out_per_sub = mod.k if mod.k <= 2 else 2
    x   = KL.Conv1D(out_per_sub, 1, activation=final_act, name='det_out')(x)
    out = KL.Reshape((N_DATA * out_per_sub,), name='bits')(x)

    model = keras.Model(inp, out, name='DetectorNet')
    n_params = model.count_params()
    n_conv = len(cfg['conv_layers'])
    loss_name = 'bce' if mod.k <= 2 else 'mse_sym'
    print(f"  {'DetectorNet':>12} | {mod_name:>8} | SIGNAL_DETECTION      | "
          f"вход={in_sz:>4} выход={out_sz:>4} loss={loss_name} "
          f"conv={n_conv} парам={n_params:,}")
    return model, in_sz, out_sz


# ================================================================
# C: СОВМЕСТНАЯ СЕТЬ (JointNet)
# ================================================================
#
# Объединяет ChanEstNet и DetectorNet в одну дифференцируемую модель.
# Преимущество: градиент ошибки детектора «протекает» через H_pred
# обратно к оценщику → оценщик настраивается на минимальную BER,
# а не только на минимальный MSE по H (что было на шаге 1).
#
# Вход: [Re(Yp_norm)*N_PILOT, Im(Yp_norm)*N_PILOT,
#         Re(Yd_norm)*N_DATA,  Im(Yd_norm)*N_DATA, snr]
#
# Выходы: [H_pred (2*N_DATA), bits_out (N_DATA*k или 2*N_DATA)]
# Loss: alpha*MSE(H_pred) + (1-alpha)*BCE/MSE(bits), alpha=0.3.
#
# КРИТИЧНО: имена слоёв оценочной ветви = ChanEstNet.
# train_joint() копирует предобученные веса по именам слоёв.
# Смотри комментарий к build_chan_est_net() выше.
# ================================================================

def build_joint_net(mod_name):
    """Строит JointNet. Ветвь детектора использует Conv1D для сохранения
    локальной корреляции поднесущих, что критично при разреженных пилотах."""
    cfg = ARCH_CONFIG['JointNet']
    mod = MODULATIONS[mod_name]
    extra = 1 if USE_SNR_FEATURE else 0
    in_pilot = 2 * N_PILOT
    in_data  = 2 * N_DATA
    in_sz    = in_pilot + in_data + extra
    out_sz_det = N_DATA * mod.k if mod.k <= 2 else 2 * N_DATA
    final_act  = 'sigmoid' if mod.k <= 2 else 'linear'
    out_per_sub = mod.k if mod.k <= 2 else 2
    
    inp = keras.Input(shape=(in_sz,), name='joint_input')

    # ── Ветвь 1: оценка канала (ИМЕНА СТРОГО = ChanEstNet для переноса весов) ──
    re_p = _Slice(0, N_PILOT, name='re_pilot')(inp)
    im_p = _Slice(N_PILOT, N_PILOT, name='im_pilot')(inp)
    grid_p = KL.Concatenate(axis=-1, name='pilot_grid')([
        KL.Reshape((N_PILOT, 1))(re_p),
        KL.Reshape((N_PILOT, 1))(im_p)])

    h = grid_p
    for i, (filters, kernel) in enumerate(cfg['est_conv_layers'], 1):
        h = KL.Conv1D(filters, kernel, padding='same', name=f'conv{i}')(h)
        h = KL.BatchNormalization(name=f'bn_c{i}')(h)
        h = KL.ReLU(name=f'relu_c{i}')(h)

    h = KL.Flatten()(h)
    if extra > 0:
        snr_f = _Slice(in_pilot + in_data, extra, name='snr_feat')(inp)
        h = KL.Concatenate()([h, snr_f])

    for i, units in enumerate(cfg['est_dense_layers'], 1):
        h = KL.Dense(units, name=f'fc{i}')(h)
        h = KL.BatchNormalization(name=f'bn_f{i}')(h)
        h = KL.ReLU(name=f'relu_f{i}')(h)

    H_pred = KL.Dense(2 * N_DATA, activation='linear', name='h_pred')(h)

    # ── Ветвь 2: детектирование (Conv1D для локальной обработки) ──
    re_y = _Slice(in_pilot, N_DATA, name='re_yd')(inp)
    im_y = _Slice(in_pilot + N_DATA, N_DATA, name='im_yd')(inp)
    re_H = _Slice(0, N_DATA, name='re_Hpred')(H_pred)
    im_H = _Slice(N_DATA, N_DATA, name='im_Hpred')(H_pred)
    
    grid_d = KL.Concatenate(axis=-1, name='det_grid')([
        KL.Reshape((N_DATA, 1))(re_y),
        KL.Reshape((N_DATA, 1))(im_y),
        KL.Reshape((N_DATA, 1))(re_H),
        KL.Reshape((N_DATA, 1))(im_H)])

    d = grid_d
    for i, (filters, kernel) in enumerate(cfg['det_conv_layers'], 1):
        d = KL.Conv1D(filters, kernel, padding='same', name=f'j_d_conv{i}')(d)
        d = KL.BatchNormalization(name=f'j_d_bn{i}')(d)
        d = KL.ReLU(name=f'j_d_relu{i}')(d)

    d = KL.Conv1D(out_per_sub, 1, activation=final_act, name='j_det_out')(d)
    bits_out = KL.Reshape((out_sz_det,), name='bits')(d)

    model = keras.Model(inp, [H_pred, bits_out], name='JointNet')
    n_params = model.count_params()
    print(f"  {'JointNet':>12} | {mod_name:>8} | END_TO_END            | "
          f"вход={in_sz:>4} выход=({2*N_DATA},{out_sz_det}) "
          f"est_conv={len(cfg['est_conv_layers'])} det_conv={len(cfg['det_conv_layers'])} парам={n_params:,}")
    return model, in_sz, out_sz_det


# ================================================================
# D–F: КЛАССИЧЕСКИЕ АРХИТЕКТУРЫ (для сравнения с DNN-методами)
# ================================================================

def _build_mlp(input_size, output_size, hidden, n_layers, final_activation='sigmoid'):
    """
    MLP — полносвязная сеть: n_layers × [Dense + BatchNorm + ReLU].
    Базовый эталон: не учитывает частотную структуру сигнала.
    """
    inp = keras.Input(shape=(input_size,), name='input')
    x   = inp
    for i in range(n_layers):
        x = KL.Dense(hidden, name=f'dense_{i}')(x)
        x = KL.BatchNormalization(name=f'bn_{i}')(x)
        x = KL.ReLU(name=f'relu_{i}')(x)
    return keras.Model(inp, KL.Dense(output_size, activation=final_activation, name='output')(x), name='MLP')


def _build_resnet(input_size, output_size, hidden, n_blocks, final_activation='sigmoid'):
    """
    ResidualNet — с остаточными (skip) связями.
    Блок: x → Dense → BN → ReLU → Dense → BN → Add(x, out) → ReLU.
    Остаточные связи решают проблему затухания градиента в глубоких сетях:
    Z = F(x) + x позволяет градиенту «перешагнуть» через блок.
    """
    inp = keras.Input(shape=(input_size,), name='input')
    x   = KL.Dense(hidden)(inp)
    x   = KL.BatchNormalization()(x); x = KL.ReLU()(x)
    for _ in range(n_blocks):
        res = x   # запоминаем «скачок» для сложения
        x   = KL.Dense(hidden)(x); x = KL.BatchNormalization()(x); x = KL.ReLU()(x)
        x   = KL.Dense(hidden)(x); x = KL.BatchNormalization()(x)
        x   = KL.Add()([x, res]); x = KL.ReLU()(x)   # Z = F(x) + x
    return keras.Model(inp, KL.Dense(output_size, activation=final_activation, name='output')(x), name='ResidualNet')


def _build_convnet(input_size, output_size, n_freq, n_filters=64, final_activation='sigmoid'):
    """
    ConvNet — свёрточная по частотной оси.
    Входной вектор [Re(X), Im(X), ...] разбивается на (n_freq, 2) для Conv1D.
    n_freq = N_PILOT (ChanEst) / N_DATA (Det) / N_FFT (E2E).
    Дополнительные признаки (SNR и т.п.) конкатенируются после Flatten.
    """
    inp  = keras.Input(shape=(input_size,), name='input')
    re   = _Slice(0,      n_freq, name='slice_re')(inp)
    im   = _Slice(n_freq, n_freq, name='slice_im')(inp)
    # Собираем (n_freq, 2): каждый «пиксель» = одна поднесущая
    grid = KL.Concatenate(axis=-1)([KL.Reshape((n_freq, 1))(re),
                                    KL.Reshape((n_freq, 1))(im)])
    x = KL.Conv1D(n_filters,   5, padding='same')(grid)
    x = KL.BatchNormalization()(x); x = KL.ReLU()(x)
    x = KL.Conv1D(n_filters*2, 3, padding='same')(x)
    x = KL.BatchNormalization()(x); x = KL.ReLU()(x)
    x = KL.Conv1D(n_filters,   3, padding='same')(x)
    x = KL.BatchNormalization()(x); x = KL.ReLU()(x)
    x = KL.Flatten()(x)
    # Дополнительные признаки (SNR) добавляем после Flatten
    extra = input_size - 2 * n_freq
    if extra > 0:
        x = KL.Concatenate()([x, _Slice(2 * n_freq, extra, name='slice_extra')(inp)])
    x = KL.Dense(256)(x); x = KL.BatchNormalization()(x); x = KL.ReLU()(x)
    return keras.Model(inp, KL.Dense(output_size, activation=final_activation, name='output')(x), name='ConvNet')


def build_model(arch_name, mod_name, research_mode=None):
    """
    Универсальная фабрика моделей. Возвращает (model, n_params, loss).

    arch_name : 'ChanEstNet'|'DetectorNet'|'JointNet'|'MLP'|'ResidualNet'|'ConvNet'
    mod_name  : модуляция из MODULATIONS
    research_mode : str или None

    Для классических архитектур (MLP/ResNet/ConvNet) размер hidden
    масштабируется с k: QAM64 (heavy=True) → hidden=512, больше слоёв.
    """
    if research_mode is None:
        research_mode = RESEARCH_MODE

    if arch_name == 'ChanEstNet':
        m, _, _ = build_chan_est_net(mod_name)
        return m, m.count_params(), 'mse'
    if arch_name == 'DetectorNet':
        m, _, _ = build_detector_net(mod_name)
        mod = MODULATIONS[mod_name]
        return m, m.count_params(), 'binary_crossentropy' if mod.k <= 2 else 'mse'
    if arch_name == 'JointNet':
        m, _, _ = build_joint_net(mod_name)
        mod = MODULATIONS[mod_name]
        return m, m.count_params(), {
            'h_pred': 'mse',
            'bits':   'binary_crossentropy' if mod.k <= 2 else 'mse'}

    mod   = MODULATIONS[mod_name]
    heavy = mod.k >= 6   # QAM64 — более сложная задача → больше нейронов
    input_size, output_size, n_freq, loss, final_act = get_io_sizes(mod_name, research_mode)

    if arch_name == 'MLP':
        m = _build_mlp(input_size, output_size,
                       hidden=512 if heavy else 256,
                       n_layers=4 if heavy else 3,
                       final_activation=final_act)
    elif arch_name == 'ResidualNet':
        m = _build_resnet(input_size, output_size,
                          hidden=512 if heavy else 256,
                          n_blocks=5 if heavy else 4,
                          final_activation=final_act)
    elif arch_name == 'ConvNet':
        m = _build_convnet(input_size, output_size,
                           n_freq=n_freq, final_activation=final_act)
    else:
        raise ValueError(f"Неизвестная архитектура: {arch_name}")

    n_params = m.count_params()
    print(f"  {arch_name:>12} | {mod_name:>8} | {research_mode:<20} | "
          f"вход={input_size:>4} выход={output_size:>4} loss={loss} параметров={n_params:,}")
    return m, n_params, loss


# ── Проверка архитектур ──────────────────────────────────────────
print(f"{'Арх.':>12} | {'Мод.':>8} | {'Режим':<20} | сводка")
print("-" * 78)
for arch in ['MLP', 'ResidualNet', 'ConvNet']:
    build_model(arch, 'BPSK')
print()
for arch in ['ChanEstNet', 'DetectorNet', 'JointNet']:
    build_model(arch, 'QPSK')

# Проверяем форму выходов JointNet
_extra = 1 if USE_SNR_FEATURE else 0
_m_j, _, _ = build_model('JointNet', 'QPSK')
_in_j = 2 * N_PILOT + 2 * N_DATA + _extra
_xj   = np.random.randn(4, _in_j).astype(np.float32)
_yj   = _m_j(_xj, training=False)
assert len(_yj) == 2
assert _yj[0].shape == (4, 2 * N_DATA)
assert _yj[1].shape == (4, N_DATA * MODULATIONS['QPSK'].k)
print(f"\nJointNet ✓  H_pred={tuple(_yj[0].shape)}  bits={tuple(_yj[1].shape)}")
print("\n✓ Блок 4 готов.")

In [ ]:
# ================================================================
# БЛОК 5: ФУНКЦИИ ОБУЧЕНИЯ
# ================================================================
#
# ТРЁХЭТАПНЫЙ ПРОТОКОЛ:
#
# Шаг 1 — train_chan_est():
#   Обучаем ChanEstNet: Y_pilot_norm → H_data_norm, loss MSE.
#   LR=3e-3 (фиксированный, не масштабированный) — для MSE быстро.
#
# Шаг 2 — train_detector():
#   Используем предобученный ChanEstNet для получения H_dnn,
#   строим Z_clip = clip(Y_data/H_dnn, ±3.0),
#   обучаем DetectorNet: [Z_clip, H_norm] → биты/символы.
#   loss: BCE (BPSK/QPSK) или MSE(символ) (QAM16/64).
#
# Шаг 3 — train_joint():
#   Строим JointNet, переносим веса ChanEstNet по именам слоёв,
#   делаем fine-tuning: loss = 0.3*MSE(H) + 0.7*BCE(bits).
#   Градиент ошибки детектора настраивает оценщик под минимальную BER.
#   LR уменьшен вдвое — тонкая настройка, а не грубое переобучение.
# ================================================================

import gc
os.makedirs('saved_models', exist_ok=True)

# Порог клиппинга ZF-сигнала. При глубоком замирании |H|→0 → |Z|→∞.
# _Z_CLIP=3.0 безопасен: |X|≈0.7 (QPSK), ≈1.08 (QAM64).
_Z_CLIP = 3.0

# Масштабирование LR по batch: √(batch/base_batch) — «квадратный корень».
# Более агрессивное линейное правило (LR*K) нестабильно при large batch.
_BASE_BATCH = 512
_BASE_LR    = 1e-3
_SCALED_LR  = _BASE_LR * np.sqrt(BATCH_SIZE / _BASE_BATCH)
_SCALED_LR  = float(np.clip(_SCALED_LR, 1e-3, 1e-2))


def _make_detector_input(g, H_dnn_flat, snr, n):
    """
    Строит входной вектор DetectorNet из generate() и предсказанного H_dnn.

    Выходной вектор: [Re(Z_clip), Im(Z_clip), Re(H_norm), Im(H_norm), (snr)]
    размерность = 4*N_DATA + extra.

    H_dnn_flat : (n, 2*N_DATA) float32 — выход ChanEstNet.
    """
    extra = 1 if USE_SNR_FEATURE else 0
    H_complex = H_dnn_flat[:, :N_DATA] + 1j * H_dnn_flat[:, N_DATA:]
    if USE_INPUT_NORMALIZATION:
        H_complex = H_complex * g['norm']   # денормализация: абсолютный масштаб
    H_safe = np.where(np.abs(H_complex) < 1e-9, 1e-9+0j, H_complex)
    Z = g['Y_data'] / H_safe   # ZF: |Z| ≈ |X| ≈ 1
    # Ограничиваем выбросы при глубоком замирании
    re_z = np.clip(np.real(Z).astype(np.float32), -_Z_CLIP, _Z_CLIP)
    im_z = np.clip(np.imag(Z).astype(np.float32), -_Z_CLIP, _Z_CLIP)
    # H_norm: индикатор качества поднесущей (|H_norm|≪1 → замирание → шум усилен)
    H_norm = H_complex / (g['norm'] + 1e-9)
    re_H   = np.real(H_norm).astype(np.float32)
    im_H   = np.imag(H_norm).astype(np.float32)
    parts  = [re_z, im_z, re_H, im_H]
    if extra > 0:
        span  = float(SNR_TRAIN_MAX - SNR_TRAIN_MIN)
        parts.append(np.full((n, 1), (snr - SNR_TRAIN_MIN) / span, dtype=np.float32))
    return np.hstack(parts).astype(np.float32)


def _get_true_symbols(g, mod):
    """
    Восстанавливает истинные комплексные символы из битов g['bits'].
    «Мягкая» цель для DetectorNet/JointNet при QAM16/64 (loss MSE).
    Возвращает (N, N_DATA) complex.
    """
    N = g['bits'].shape[0]
    return mod.modulate(g['bits'].reshape(N * N_DATA, mod.k)).reshape(N, N_DATA)


def _decode_interleaved_symbols(out):
    """
    Декодирует перемежённый выход DetectorNet/JointNet (QAM16/64).

    Conv1D(2, kernel=1) + Reshape даёт порядок:
      [Re_s0, Im_s0, Re_s1, Im_s1, ..., Re_s(N_DATA-1), Im_s(N_DATA-1)]
    (чётные индексы → Re, нечётные → Im)

    out : (N, 2*N_DATA) float32 → (N, N_DATA) complex.
    """
    return out[:, 0::2] + 1j * out[:, 1::2]


def _build_training_data_chan_est(mod_name, n_total):
    """
    Данные для ChanEstNet: X=[Re(Yp_norm), Im(Yp_norm), snr], y=[Re(H_norm), Im(H_norm)].
    Равномерно по 20 точкам SNR: модель видит все условия при обучении.
    """
    n_pts = 20
    grid  = np.linspace(SNR_TRAIN_MIN, SNR_TRAIN_MAX, n_pts)
    chunk = max(1, n_total // n_pts)
    X_all, y_all = [], []
    for snr in grid:
        g = generate(chunk, float(snr), mod_name)
        Xc, yc = build_xy(g, research_mode='CHANNEL_ESTIMATION')
        X_all.append(Xc); y_all.append(yc)
    return np.vstack(X_all), np.vstack(y_all)


def _build_training_data_detector(mod_name, chan_est_model, n_total):
    """
    Данные для DetectorNet. Требует предобученный chan_est_model (шаг 1).
    Один батчевый predict() для всех SNR — без повторной трассировки TF-графа.

    Цель:
        k≤2: биты float32 (loss BCE)
        k≥4: символы Re/Im в ПЕРЕМЕЖЁННОМ порядке (loss MSE)
    ВАЖНО: порядок [Re_s0, Im_s0, Re_s1, ...] должен совпадать
    с порядком выхода Conv1D+Reshape в build_detector_net().
    """
    mod   = MODULATIONS[mod_name]
    n_pts = 20
    grid  = np.linspace(SNR_TRAIN_MIN, SNR_TRAIN_MAX, n_pts)
    chunk = max(1, n_total // n_pts)
    gs, Xp_all = [], []
    for snr in grid:
        g = generate(chunk, float(snr), mod_name)
        Xp, _ = build_xy(g, research_mode='CHANNEL_ESTIMATION')
        gs.append(g); Xp_all.append(Xp)
    H_dnn_all = chan_est_model.predict(np.vstack(Xp_all), batch_size=BATCH_SIZE, verbose=0)
    X_all, y_all = [], []
    for i, (g, snr) in enumerate(zip(gs, grid)):
        sl = slice(i * chunk, (i + 1) * chunk)
        X  = _make_detector_input(g, H_dnn_all[sl], snr, chunk)
        if mod.k >= 4:
            X_true = _get_true_symbols(g, mod)
            y = np.empty((chunk, 2 * N_DATA), dtype=np.float32)
            y[:, 0::2] = np.real(X_true)   # Re на чётных позициях
            y[:, 1::2] = np.imag(X_true)   # Im на нечётных позициях
        else:
            y = g['bits'].astype(np.float32)
        X_all.append(X); y_all.append(y)
    return np.vstack(X_all), np.vstack(y_all)


def _build_training_data_joint(mod_name, n_total):
    """
    Данные для JointNet.
    Вход: [Re(Yp_norm), Im(Yp_norm), Re(Yd_norm), Im(Yd_norm), snr].
    JointNet принимает сырой Y_data (не Z): деление Y/H_pred делается
    внутри дифференцируемого графа.
    Цели: {'h_pred': [Re(H_norm), Im(H_norm)], 'bits': биты_или_символы}.
    """
    mod   = MODULATIONS[mod_name]
    extra = 1 if USE_SNR_FEATURE else 0
    n_pts = 20
    grid  = np.linspace(SNR_TRAIN_MIN, SNR_TRAIN_MAX, n_pts)
    chunk = max(1, n_total // n_pts)
    X_all, yH_all, yB_all = [], [], []
    for snr in grid:
        g = generate(chunk, float(snr), mod_name)
        Yp = g['Y_pilot'] / (g['norm'] if USE_INPUT_NORMALIZATION else 1)
        Yd = g['Y_data']  / (g['norm'] if USE_INPUT_NORMALIZATION else 1)
        parts = [np.real(Yp).astype(np.float32), np.imag(Yp).astype(np.float32),
                 np.real(Yd).astype(np.float32), np.imag(Yd).astype(np.float32)]
        if extra > 0:
            span = float(SNR_TRAIN_MAX - SNR_TRAIN_MIN)
            parts.append(np.full((chunk, 1), (snr - SNR_TRAIN_MIN) / span, dtype=np.float32))
        X  = np.hstack(parts).astype(np.float32)
        Ht = g['H_true_data'] / (g['norm'] if USE_INPUT_NORMALIZATION else 1)
        yH = np.hstack([np.real(Ht), np.imag(Ht)]).astype(np.float32)
        if mod.k >= 4:
            X_true = _get_true_symbols(g, mod)
            yB = np.empty((chunk, 2 * N_DATA), dtype=np.float32)
            yB[:, 0::2] = np.real(X_true); yB[:, 1::2] = np.imag(X_true)
        else:
            yB = g['bits'].astype(np.float32)
        X_all.append(X); yH_all.append(yH); yB_all.append(yB)
    return np.vstack(X_all), np.vstack(yH_all), np.vstack(yB_all)


def _warmup_callbacks(patience=20, lr_patience=7):
    """
    Callbacks для model.fit():

    ReduceLROnPlateau: если train_loss не улучшается lr_patience эпох —
      умножает LR на 0.5. Автоматически «пробивает» плато.
      min_lr=1e-5: LR не опустится ниже этого значения.

    EarlyStopping: если loss не улучшается patience эпох — останавливает.
      restore_best_weights=True: возвращает лучшие веса (по min loss),
      а не последние (которые могут быть хуже из-за переобучения).
    """
    return [
        keras.callbacks.ReduceLROnPlateau(
            monitor='loss', factor=0.5, patience=lr_patience,
            min_lr=1e-5, verbose=0),
        keras.callbacks.EarlyStopping(
            monitor='loss', patience=patience,
            restore_best_weights=True, verbose=0),
    ]


def _model_path(arch_name, mod_name, stage=''):
    """Путь к файлу модели внутри папки текущего эксперимента."""
    tag = f'_{stage}' if stage else ''
    filename = f"{mod_name}_{arch_name}{tag}.keras"
    return os.path.join(BASE_SAVE_DIR, filename)


def load_model_if_exists(arch_name, mod_name, stage=''):
    """
    Загружает модель из файла если он существует.
    Возвращает (model, True) или (None, False).
    custom_objects={'_Slice': _Slice} обязателен: при загрузке Keras
    должен знать пользовательский слой для восстановления графа.
    """
    path = _model_path(arch_name, mod_name, stage)
    if os.path.exists(path):
        m = keras.models.load_model(path, custom_objects={'_Slice': _Slice})
        print(f"  Загружена: {path}")
        return m, True
    return None, False


def train_chan_est(mod_name, verbose=True, save=True):
    """
    Шаг 1/3: обучает ChanEstNet (MSE, Y_pilot_norm → H_data_norm).
    После обучения X/y удаляются: при N_TRAIN=1M занимают ~120MB+ RAM.
    """
    if verbose:
        print(f"\n{'='*60}")
        print(f"  [1/3] Оценка канала: {mod_name}  (MSE)  LR=3e-3  BATCH={BATCH_SIZE}")
        print(f"{'='*60}")
    model, _, _ = build_model('ChanEstNet', mod_name)
    model.compile(optimizer=keras.optimizers.Adam(3e-3), loss='mse')
    X, y = _build_training_data_chan_est(mod_name, N_TRAIN)
    if verbose: print(f"  Данные: X{X.shape}  y{y.shape}")
    hist = model.fit(X, y, epochs=150, batch_size=BATCH_SIZE,
                     callbacks=_warmup_callbacks(patience=25, lr_patience=10),
                     shuffle=True, verbose=1 if verbose else 0)
    del X, y; gc.collect()
    if verbose:
        print(f"  Готово: {len(hist.history['loss'])} эпох,  MSE={hist.history['loss'][-1]:.5f}")
    if save:
        path = _model_path('ChanEstNet', mod_name, 'step1')
        model.save(path); print(f"  Сохранена: {path}")
    return model, hist.history['loss']


def train_detector(mod_name, chan_est_model, verbose=True, save=True):
    """
    Шаг 2/3: обучает DetectorNet ([Z_clip, H_norm] → биты/символы).
    Требует предобученный chan_est_model (шаг 1).
    n_epochs масштабируется с k: QAM64 сложнее → больше эпох.
    """
    mod = MODULATIONS[mod_name]
    if verbose:
        loss_name = 'MSE(символ)' if mod.k >= 4 else 'BCE(биты)'
        print(f"\n{'='*60}")
        print(f"  [2/3] Детектор: {mod_name}  ({loss_name})  LR={_SCALED_LR:.5f}")
        print(f"{'='*60}")
    model, _, _ = build_model('DetectorNet', mod_name)
    loss_fn = 'mse' if mod.k >= 4 else 'binary_crossentropy'
    model.compile(optimizer=keras.optimizers.Adam(_SCALED_LR), loss=loss_fn)
    n_epochs = {1: 100, 2: 150, 4: 200, 6: 300}.get(mod.k, 150)
    if verbose: print(f"  Генерация данных (N={N_TRAIN})...")
    X, y = _build_training_data_detector(mod_name, chan_est_model, N_TRAIN)
    if verbose: print(f"  Данные: X{X.shape}  y{y.shape}  loss={loss_fn}")
    hist = model.fit(X, y, epochs=n_epochs, batch_size=BATCH_SIZE,
                     callbacks=_warmup_callbacks(patience=30, lr_patience=10),
                     shuffle=True, verbose=1 if verbose else 0)
    del X, y; gc.collect()
    if verbose:
        print(f"  Готово: {len(hist.history['loss'])} эпох,  loss={hist.history['loss'][-1]:.5f}")
    if save:
        path = _model_path('DetectorNet', mod_name, 'step2')
        model.save(path); print(f"  Сохранена: {path}")
    return model, hist.history['loss']


def train_joint(mod_name, chan_est_model=None, verbose=True, save=True, alpha=0.3):
    """
    Шаг 3/3: JointNet.
    
    Если JOINT_TRANSFER_WEIGHTS = False:
        Обучение с нуля. Сеть сама находит оптимальную стратегию
        для нелинейного канала. Требует больше данных и эпох.
    
    Если JOINT_TRANSFER_WEIGHTS = True:
        Перенос весов из ChanEstNet. Быстрее сходится, но может
        застрять в локальном минимуме, унаследованном от линейного
        режима обучения ChanEstNet.
    """
    mod = MODULATIONS[mod_name]
    loss_det = 'mse' if mod.k >= 4 else 'binary_crossentropy'
    transfer = JOINT_TRANSFER_WEIGHTS
    
    if verbose:
        mode_str = "С ПЕРЕНОСОМ весов" if transfer else "С НУЛЯ (без переноса)"
        print(f"\n{'='*60}")
        print(f"  [3/3] JointNet: {mod_name}  ({alpha:.1f}*MSE_H + {1-alpha:.1f}*{loss_det})")
        print(f"  Режим: {mode_str}")
        print(f"  LR={_SCALED_LR*0.5:.5f}  BATCH={BATCH_SIZE}  EPOCHS={JOINT_EPOCHS}")
        print(f"{'='*60}")
    
    model, _, _ = build_model('JointNet', mod_name)
    
    # ── Перенос весов (если включён) ──────────────────────────
    if transfer and chan_est_model is not None:
        transferred = 0
        for layer in chan_est_model.layers:
            if not layer.get_weights():
                continue
            try:
                jl = model.get_layer(layer.name)
                jl.set_weights(layer.get_weights())
                transferred += 1
            except Exception:
                pass
        if verbose:
            print(f"  Веса ChanEstNet перенесены: {transferred} слоёв")
            if transferred == 0:
                print("  ПРЕДУПРЕЖДЕНИЕ: ни один слой не скопирован — проверьте имена!")
    elif verbose:
        print("  Веса НЕ переносятся — обучение с нуля")
    
    # ── Компиляция ─────────────────────────────────────────────
    # При обучении с нуля используем полный LR, при fine-tuning — уменьшенный
    lr = _SCALED_LR * (1.0 if not transfer else 0.5)
    
    model.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss={'h_pred': 'mse', 'bits': loss_det},
        loss_weights={'h_pred': alpha, 'bits': 1.0 - alpha},
    )
    
    # ── Данные ─────────────────────────────────────────────────
    n_samples = JOINT_TRAIN_SAMPLES if not transfer else N_TRAIN // 2
    if verbose:
        print(f"  Генерация данных (N={n_samples:,})...")
    X, yH, yB = _build_training_data_joint(mod_name, n_samples)
    if verbose:
        print(f"  Данные: X{X.shape}  yH{yH.shape}  yB{yB.shape}")
    
    # ── Обучение ───────────────────────────────────────────────
    hist = model.fit(
        X, {'h_pred': yH, 'bits': yB},
        epochs=JOINT_EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=_warmup_callbacks(patience=30, lr_patience=10),
        shuffle=True,
        verbose=1 if verbose else 0,
    )
    del X, yH, yB; gc.collect()
    
    if verbose:
        print(f"  Готово: {len(hist.history['loss'])} эпох, "
              f"loss={hist.history['loss'][-1]:.5f}")
    
    if save:
        path = _model_path('JointNet', mod_name, 'step3')
        model.save(path)
        print(f"  Сохранена: {path}")
    
    return model, hist.history['loss']


def train(arch_name, mod_name, research_mode=None, n_epochs=None, verbose=True, save=True):
    """
    Классическое одноэтапное обучение для MLP/ResNet/ConvNet.
    Данные генерируются через build_xy() с текущим RESEARCH_MODE.
    """
    if research_mode is None: research_mode = RESEARCH_MODE
    mod = MODULATIONS[mod_name]
    if n_epochs is None:
        n_epochs = {1: 80, 2: 120, 4: 200, 6: 300}.get(mod.k, 120)
    model, n_params, loss = build_model(arch_name, mod_name, research_mode)
    n_pts = 20; grid = np.linspace(SNR_TRAIN_MIN, SNR_TRAIN_MAX, n_pts)
    chunk = max(1, N_TRAIN // n_pts)
    X_all, y_all = [], []
    for snr in grid:
        g = generate(chunk, float(snr), mod_name)
        Xc, yc = build_xy(g, research_mode)
        X_all.append(Xc); y_all.append(yc)
    model.compile(optimizer=keras.optimizers.Adam(_SCALED_LR), loss=loss)
    hist = model.fit(np.vstack(X_all), np.vstack(y_all),
                     epochs=n_epochs, batch_size=BATCH_SIZE,
                     callbacks=_warmup_callbacks(), shuffle=True,
                     verbose=1 if verbose else 0)
    if save:
        model.save(_model_path(arch_name, mod_name))
    return model, hist.history['loss'], n_params


# ── Быстрый тест ─────────────────────────────────────────────────
print(f"\n✓ Блок 5 готов.  LR масштабированный = {_SCALED_LR:.5f}  (BATCH={BATCH_SIZE})")

In [ ]:
# ================================================================
# БЛОК 6: ДЕТЕКТОРЫ И ВЫЧИСЛЕНИЕ BER
# ================================================================
#
# detect_ls()          — LS/ZF: компенсируем DFT-оценку H делением
# detect_mmse()        — MMSE: взвешенная эквализация W = H*/(|H|²+σ²)
# detect_dnn_two_stage() — ChanEstNet → DetectorNet (два predict)
# detect_dnn_joint()   — JointNet (один predict)
# detect_dnn()         — универсальный для MLP/ResNet/ConvNet
#
# compute_ber_oracle() — нижняя граница BER (истинный H, MMSE)
# compute_ber()        — базовая BER-функция (для LS/MMSE)
# compute_ber_two_stage() / compute_ber_joint() — оптимизированные:
#   последовательная обработка SNR + явная очистка RAM после каждой точки
#   (предотвращает накопление 80×n_samples объектов в памяти).
# ================================================================


def detect_ls(g, mod_name, snr_dB=None):
    """
    LS/ZF-детектор: Z = Y_data / H_est, затем ML-детектирование.
    Полностью убирает H, но усиливает шум на слабых поднесущих
    (noise enhancement). snr_dB не используется.
    """
    mod  = MODULATIONS[mod_name]
    Hs   = np.where(np.abs(g['H_est_data']) < 1e-9, 1e-9+0j, g['H_est_data'])
    eq   = (g['Y_data'] / Hs).reshape(-1)
    return mod.demodulate(eq).reshape(g['bits'].shape[0], N_DATA * mod.k)


def detect_mmse(g, mod_name, snr_dB=None):
    """
    MMSE-детектор: W = conj(H) / (|H|² + σ²), σ² = 1/SNR_lin.
    Компромисс: при малом |H| ограничивает усиление шума в ущерб
    полной компенсации канала. При высоком SNR → вырождается в ZF.
    Теоретически всегда ≥ ZF по качеству (критерий MSE).
    """
    mod    = MODULATIONS[mod_name]
    H      = g['H_est_data']
    sigma2 = 1.0 / (10.0 ** (snr_dB / 10.0)) if snr_dB is not None else 0.01
    W   = np.conj(H) / (np.abs(H) ** 2 + sigma2)
    eq  = (W * g['Y_data']).reshape(-1)
    return mod.demodulate(eq).reshape(g['bits'].shape[0], N_DATA * mod.k)


def detect_dnn_two_stage(g, chan_est_model, detector_model, mod_name):
    """
    Двухэтапный DNN: шаг 1 — ChanEstNet(Y_pilot_norm)→H_dnn,
    шаг 2 — DetectorNet([Z_clip, H_norm])→биты/символы.
    QAM16/64: выход сети — символы Re/Im → demodulate().
    """
    mod = MODULATIONS[mod_name]
    N   = g['bits'].shape[0]
    Xp, _ = build_xy(g, research_mode='CHANNEL_ESTIMATION')
    H_dnn_flat = chan_est_model.predict(Xp, batch_size=BATCH_SIZE, verbose=0)
    X_det = _make_detector_input(g, H_dnn_flat, g['snr_dB'], N)
    pred  = detector_model.predict(X_det, batch_size=BATCH_SIZE, verbose=0)
    if mod.k >= 4:
        X_est = _decode_interleaved_symbols(pred)
        return mod.demodulate(X_est.reshape(-1)).reshape(N, N_DATA*mod.k).astype(np.int32)
    return (pred > 0.5).astype(np.int32)


def detect_dnn_joint(g, joint_model, mod_name):
    """
    JointNet: один predict() → берём второй выход (bits_out).
    Формирует [Y_pilot_norm, Y_data_norm, snr] как при обучении.
    """
    mod   = MODULATIONS[mod_name]
    extra = 1 if USE_SNR_FEATURE else 0
    N     = g['bits'].shape[0]
    Yp = g['Y_pilot'] / (g['norm'] if USE_INPUT_NORMALIZATION else 1)
    Yd = g['Y_data']  / (g['norm'] if USE_INPUT_NORMALIZATION else 1)
    parts = [np.real(Yp).astype(np.float32), np.imag(Yp).astype(np.float32),
             np.real(Yd).astype(np.float32), np.imag(Yd).astype(np.float32)]
    if extra > 0:
        span = float(SNR_TRAIN_MAX - SNR_TRAIN_MIN)
        parts.append(np.full((N, 1), (g['snr_dB'] - SNR_TRAIN_MIN) / span, dtype=np.float32))
    _, det_out = joint_model.predict(np.hstack(parts).astype(np.float32),
                                     batch_size=BATCH_SIZE, verbose=0)
    if mod.k >= 4:
        X_est = _decode_interleaved_symbols(det_out)
        return mod.demodulate(X_est.reshape(-1)).reshape(N, N_DATA*mod.k).astype(np.int32)
    return (det_out > 0.5).astype(np.int32)


def detect_dnn(g, model, mod_name, research_mode=None):
    """
    Универсальный детектор для MLP/ResNet/ConvNet.
    CHANNEL_ESTIMATION: model→H_pred → MMSE-эквализация → demodulate.
    SIGNAL_DETECTION / END_TO_END: model→биты → порог 0.5.
    """
    if research_mode is None: research_mode = RESEARCH_MODE
    mod  = MODULATIONS[mod_name]
    X, _ = build_xy(g, research_mode)
    pred = model.predict(X, batch_size=BATCH_SIZE, verbose=0)
    if research_mode == 'CHANNEL_ESTIMATION':
        H_pred = pred[:, :N_DATA] + 1j * pred[:, N_DATA:]
        if USE_INPUT_NORMALIZATION: H_pred = H_pred * g['norm']
        Hs = np.where(np.abs(H_pred) < 1e-9, 1e-9+0j, H_pred)
        sigma2 = 1.0 / (10.0 ** (g['snr_dB'] / 10.0))
        W  = np.conj(Hs) / (np.abs(Hs)**2 + sigma2)
        eq = (W * g['Y_data']).reshape(-1)
        return mod.demodulate(eq).reshape(g['bits'].shape[0], N_DATA * mod.k)
    return (pred > 0.5).astype(np.int32)


def compute_ber_oracle(mod_name, snr_range=SNR_TEST, n_samples=N_TEST):
    """
    Oracle: MMSE с ИСТИННЫМ H. Нижняя граница BER — ошибки только от шума.
    Разрыв Oracle — реальный метод = цена ошибок оценки канала.
    """
    mod  = MODULATIONS[mod_name]
    bers = []
    for snr in snr_range:
        g      = generate(n_samples, float(snr), mod_name)
        H      = g['H_true_data']   # истинный H!
        sigma2 = 1.0 / (10.0 ** (snr / 10.0))
        W      = np.conj(H) / (np.abs(H)**2 + sigma2)
        pred   = mod.demodulate((W * g['Y_data']).reshape(-1))
        bers.append(float(np.mean(pred.reshape(n_samples, -1) != g['bits'])))
        del g, H, W, pred; gc.collect()
    return np.array(bers)


def compute_ber(det_fn, mod_name, snr_range=SNR_TEST, n_samples=N_TEST):
    """Базовая BER-функция. det_fn(g, mod_name, snr) → bits_pred."""
    bers = []
    for snr in snr_range:
        g = generate(n_samples, float(snr), mod_name)
        pred_bits = det_fn(g, mod_name, snr)
        bers.append(float(np.mean(pred_bits != g['bits'])))
        del g, pred_bits; gc.collect()
    return np.array(bers)


def compute_ber_dnn(model, mod_name, snr_range=SNR_TEST, n_samples=N_TEST, research_mode=None):
    """BER для MLP/ResNet/ConvNet через detect_dnn()."""
    def _d(g, m, snr): return detect_dnn(g, model, m, research_mode)
    return compute_ber(_d, mod_name, snr_range, n_samples)


def compute_ber_two_stage(chan_est_model, detector_model, mod_name,
                          snr_range=SNR_TEST, n_samples=N_TEST):
    """
    Оптимизированная BER для двухэтапного DNN.
    Последовательная обработка SNR + явное удаление массивов.
    Предотвращает накопление ~80×n_samples объектов в RAM.
    PRED_BATCH < BATCH_SIZE: меньше пиковое потребление VRAM при inference.
    """
    mod = MODULATIONS[mod_name]
    PRED_BATCH = min(BATCH_SIZE, 4096)
    bers = []
    for snr in snr_range:
        g = generate(n_samples, float(snr), mod_name)
        Xp, _ = build_xy(g, 'CHANNEL_ESTIMATION')
        H_dnn_flat = chan_est_model.predict(Xp, batch_size=PRED_BATCH, verbose=0); del Xp
        X_det = _make_detector_input(g, H_dnn_flat, snr, n_samples); del H_dnn_flat
        pred  = detector_model.predict(X_det, batch_size=PRED_BATCH, verbose=0); del X_det
        if mod.k >= 4:
            X_est = _decode_interleaved_symbols(pred)
            bits  = mod.demodulate(X_est.reshape(-1)).reshape(n_samples, N_DATA*mod.k)
        else:
            bits = (pred > 0.5).astype(np.int32)
        bers.append(float(np.mean(bits != g['bits'])))
        del g, pred, bits
        if mod.k >= 4: del X_est
        gc.collect()
    return np.array(bers)


def compute_ber_joint(joint_model, mod_name, snr_range=SNR_TEST, n_samples=N_TEST):
    """
    Оптимизированная BER для JointNet.
    Аналогична compute_ber_two_stage — последовательно + явная очистка.
    """
    mod   = MODULATIONS[mod_name]
    extra = 1 if USE_SNR_FEATURE else 0
    span  = float(SNR_TRAIN_MAX - SNR_TRAIN_MIN)
    PRED_BATCH = min(BATCH_SIZE, 4096)
    bers = []
    for snr in snr_range:
        g  = generate(n_samples, float(snr), mod_name)
        Yp = g['Y_pilot'] / (g['norm'] if USE_INPUT_NORMALIZATION else 1)
        Yd = g['Y_data']  / (g['norm'] if USE_INPUT_NORMALIZATION else 1)
        parts = [np.real(Yp).astype(np.float32), np.imag(Yp).astype(np.float32),
                 np.real(Yd).astype(np.float32), np.imag(Yd).astype(np.float32)]
        if extra > 0:
            parts.append(np.full((n_samples, 1), (snr - SNR_TRAIN_MIN) / span, dtype=np.float32))
        X_in = np.hstack(parts).astype(np.float32); del Yp, Yd, parts
        _, det_out = joint_model.predict(X_in, batch_size=PRED_BATCH, verbose=0); del X_in
        if mod.k >= 4:
            X_est = _decode_interleaved_symbols(det_out)
            bits  = mod.demodulate(X_est.reshape(-1)).reshape(n_samples, N_DATA*mod.k)
        else:
            bits = (det_out > 0.5).astype(np.int32)
        bers.append(float(np.mean(bits != g['bits'])))
        del g, det_out, bits
        if mod.k >= 4: del X_est
        gc.collect()
    return np.array(bers)

def detect_iterative(g, mod_name, snr_dB=None, n_iterations=3):
    """
    Итеративное подавление нелинейных искажений усилителя мощности (PA).
    Алгоритм:
    1. Оценить канал и детектировать символы (MMSE).
    2. Реконструировать переданный сигнал во временной области.
    3. Пропустить через модель PA для оценки нелинейных искажений.
    4. Вычесть оцененные искажения из принятого сигнала.
    5. Повторить.
    """
    mod = MODULATIONS[mod_name]
    N = g['bits'].shape[0]
    
    # Исходный принятый сигнал
    Y_freq_orig = g['Y_freq'].copy()
    
    # Параметры PA (должны совпадать с generate())
    A_sat = (1.0 / np.sqrt(N_FFT)) * 10.0 ** (PA_IBO_DB / 20.0)
    p_rapp = PA_RAPP_P
    
    # Начальное приближение: считаем, что искажений нет
    Y_freq_current = Y_freq_orig.copy()
    
    sigma2 = 1.0 / (10.0 ** (snr_dB / 10.0)) if snr_dB is not None else 0.01
    
    # Буфер для реконструированного сигнала
    X_hat_freq = np.zeros((N, N_FFT), dtype=complex)
    X_hat_freq[:, PILOT_IDX] = 1.0 + 0j # Пилоты известны заранее
    
    bits_hat = np.zeros((N, N_DATA * mod.k), dtype=np.int32)
    
    for it in range(n_iterations):
        # 1. Оценка канала по текущему (очищенному) сигналу
        Y_pilot_curr = Y_freq_current[:, PILOT_IDX]
        h_alias = np.fft.ifft(Y_pilot_curr, n=N_PILOT, axis=1)
        h_trunc = np.zeros_like(h_alias)
        h_trunc[:, :_N_EFF_TAPS] = h_alias[:, :_N_EFF_TAPS]
        H_est_full = np.fft.fft(h_trunc, n=N_FFT, axis=1)
        H_est_data = H_est_full[:, DATA_IDX]
        
        # 2. MMSE-эквализация и детектирование
        Y_data_curr = Y_freq_current[:, DATA_IDX]
        H_safe = np.where(np.abs(H_est_data) < 1e-9, 1e-9+0j, H_est_data)
        W = np.conj(H_safe) / (np.abs(H_safe)**2 + sigma2)
        Z = W * Y_data_curr
        
        # Демодуляция (получаем биты)
        bits_hat = mod.demodulate(Z.reshape(-1)).reshape(N, N_DATA * mod.k)
        
        # Если это последняя итерация, возвращаем биты
        if it == n_iterations - 1:
            break
            
        # 3. Реконструкция переданных символов
        X_hat_data = mod.modulate(bits_hat.reshape(-1, mod.k)).reshape(N, N_DATA)
        X_hat_freq[:, DATA_IDX] = X_hat_data
        
        # 4. Переход во временную область
        X_hat_time = np.fft.ifft(X_hat_freq, n=N_FFT, axis=1)
        
        # 5. Применение модели PA (Rapp) для оценки искажений
        amp = np.abs(X_hat_time)
        amp_out = amp / ((1.0 + (amp/A_sat)**(2*p_rapp))**(1.0/(2*p_rapp)))
        X_hat_time_pa = amp_out * np.exp(1j * np.angle(X_hat_time))
        
        # 6. Переход в частотную область и вычисление аддитивных искажений
        X_hat_pa_freq = np.fft.fft(X_hat_time_pa, n=N_FFT, axis=1)
        D_hat_freq = X_hat_pa_freq - X_hat_freq  # Оценка нелинейной помехи
        
        # 7. Вычитание помехи из ИСХОДНОГО принятого сигнала
        # Y_clean = Y_orig - D_hat * H_est
        Y_freq_current = Y_freq_orig - D_hat_freq * H_est_full
        
    return bits_hat

def compute_ber_iterative(mod_name, snr_range=SNR_TEST, n_samples=N_TEST, n_iterations=3):
    """BER для итеративного подавления нелинейности."""
    bers = []
    for snr in snr_range:
        g = generate(n_samples, float(snr), mod_name)
        pred_bits = detect_iterative(g, mod_name, snr, n_iterations=n_iterations)
        bers.append(float(np.mean(pred_bits != g['bits'])))
        del g, pred_bits; gc.collect()
    return np.array(bers)


# ── Быстрый тест детекторов ──────────────────────────────────────
_g = generate(200, 10.0, 'BPSK')
print(f"Тест (BPSK, SNR=10dB):  LS BER={np.mean(detect_ls(_g,'BPSK',10.)!=_g['bits']):.4f}  "
      f"MMSE BER={np.mean(detect_mmse(_g,'BPSK',10.)!=_g['bits']):.4f}")
print(f"✓ Блок 6 готов.")

In [ ]:
# ================================================================
# БЛОК 6.5: ИЗМЕРЕНИЕ ПРОИЗВОДИТЕЛЬНОСТИ (FLOPs + ВРЕМЯ)
# ================================================================
# Сравнивает вычислительную сложность (FLOPs) и реальное время
# выполнения (latency) для классических и нейросетевых детекторов.

# Метрики:
# - FLOPs: теоретическое количество операций с плавающей точкой
# - Inference time: среднее время обработки одного OFDM-символа
# - Throughput: количество символов в секунду
# ================================================================
import time
import math

def _flops_complex_mult():
    """Комплексное умножение: (a+bi)(c+di) = (ac-bd) + (ad+bc)i"""
    return 6  # 4 умножения + 2 сложения

def _flops_complex_div():
    """Комплексное деление: (a+bi)/(c+di)"""
    return 12  # 6 умножений + 4 сложения + 2 (знаменатель)

def _flops_complex_add():
    return 2

def _flops_abs_sq():
    """|H|² = Re² + Im²"""
    return 3  # 2 умножения + 1 сложение

def _flops_fft(N):
    """Complex FFT: ~5·N·log₂(N) операций (Split-radix алгоритм)"""
    if N <= 1: return 0
    return int(5 * N * math.log2(N))

def count_flops_classical(method='LS', mod_name='QPSK'):
    """
    Подсчёт FLOPs для классического детектора на ОДИН OFDM-символ.
    method: 'LS' или 'MMSE'
    mod_name: имя модуляции (для ML-детектора)
    """
    mod = MODULATIONS[mod_name]
    M = mod.M
    total = 0
    
    # 1. DFT-based оценка канала
    total += _flops_fft(N_PILOT)  # IFFT от Y_pilot
    total += _flops_fft(N_FFT)    # FFT для интерполяции
    
    # 2. Эквализация N_DATA поднесущих
    if method == 'LS':
        total += N_DATA * _flops_complex_div()  # Z = Y / H
    elif method == 'MMSE':
        for _ in range(N_DATA):
            total += _flops_abs_sq()      # |H|²
            total += 1                     # + σ²
            total += _flops_complex_mult() # conj(H)*Y
            total += _flops_complex_div()  # деление
    
    # 3. ML-детектирование: M евклидовых расстояний на поднесущую
    total += N_DATA * M * 7  # 7 FLOPs на точку
    
    return int(total)


def count_flops_dnn(model, include_bn_relu=True):
    """
    Подсчёт FLOPs для нейросети Keras на ОДИН forward pass.
    """
    total_flops = 0
    layer_details = []
    
    for layer in model.layers:
        layer_flops = 0
        layer_type = type(layer).__name__
        
        if layer_type == 'Dense':
            input_shape = layer.input_shape
            if isinstance(input_shape, list):
                in_size = input_shape[0][-1]
            else:
                in_size = input_shape[-1]
            out_size = layer.units
            layer_flops += 2 * in_size * out_size
            if layer.use_bias:
                layer_flops += out_size
        
        elif layer_type == 'Conv1D':
            input_shape = layer.input_shape
            if isinstance(input_shape, list):
                in_shape = input_shape[0]
            else:
                in_shape = input_shape
            seq_len = in_shape[1]
            c_in = in_shape[2]
            c_out = layer.filters
            k = layer.kernel_size[0]
            layer_flops += 2 * seq_len * k * c_in * c_out
            if layer.use_bias:
                layer_flops += seq_len * c_out
        
        elif layer_type == 'BatchNormalization' and include_bn_relu:
            input_shape = layer.input_shape
            if isinstance(input_shape, list):
                n_elements = 1
                for dim in input_shape[0][1:]:
                    if dim: n_elements *= dim
            else:
                n_elements = 1
                for dim in input_shape[1:]:
                    if dim: n_elements *= dim
            layer_flops += 4 * n_elements
        
        elif layer_type == 'ReLU' and include_bn_relu:
            input_shape = layer.input_shape
            if isinstance(input_shape, list):
                n_elements = 1
                for dim in input_shape[0][1:]:
                    if dim: n_elements *= dim
            else:
                n_elements = 1
                for dim in input_shape[1:]:
                    if dim: n_elements *= dim
            layer_flops += n_elements
        
        elif layer_type == 'Add':
            input_shapes = layer.input_shape
            n_elements = 1
            for dim in input_shapes[0][1:]:
                if dim: n_elements *= dim
            layer_flops += n_elements
        
        elif layer_type in ['Reshape', 'Flatten', 'InputLayer', '_Slice']:
            layer_flops = 0
        
        if layer_flops > 0:
            total_flops += layer_flops
            layer_details.append((layer.name, layer_type, layer_flops))
    
    return int(total_flops), layer_details


def format_flops(n):
    """Форматирует число FLOPs в читаемый вид."""
    if n >= 1e9:
        return f"{n/1e9:.2f} GFLOPs"
    elif n >= 1e6:
        return f"{n/1e6:.2f} MFLOPs"
    elif n >= 1e3:
        return f"{n/1e3:.2f} kFLOPs"
    else:
        return f"{n} FLOPs"


def format_time(seconds):
    """Форматирует время в читаемый вид."""
    if seconds >= 1.0:
        return f"{seconds:.3f} s"
    elif seconds >= 1e-3:
        return f"{seconds*1e3:.2f} ms"
    elif seconds >= 1e-6:
        return f"{seconds*1e6:.1f} µs"
    else:
        return f"{seconds*1e9:.0f} ns"


def measure_inference_time(detector_fn, mod_name, snr_dB=10.0, 
                          n_warmup=10, n_runs=100, n_samples=1000):
    """
    Измеряет среднее время выполнения детектора на один OFDM-символ.
    
    detector_fn: функция детектора (detect_ls, detect_mmse, detect_dnn_*)
    mod_name: имя модуляции
    snr_dB: SNR для тестовых данных
    n_warmup: количество "прогревочных" запусков (не учитываются)
    n_runs: количество замеров для усреднения
    n_samples: количество OFDM-символов в одном запуске
    
    Возвращает: (mean_time_per_symbol, std_time_per_symbol) в секундах
    """
    # Генерируем тестовые данные
    g = generate(n_samples, snr_dB, mod_name)
    
    # Прогрев (warmup) - первые запуски могут быть медленнее
    for _ in range(n_warmup):
        _ = detector_fn(g, mod_name, snr_dB)
    
    # Замеры времени
    times = []
    for _ in range(n_runs):
        start = time.perf_counter()
        _ = detector_fn(g, mod_name, snr_dB)
        end = time.perf_counter()
        times.append((end - start) / n_samples)  # время на один символ
    
    times = np.array(times)
    return float(np.mean(times)), float(np.std(times))


def measure_dnn_inference_time(model, mod_name, research_mode=None,
                               n_warmup=10, n_runs=100, n_samples=1000):
    """
    Измеряет время выполнения нейросетевого детектора.
    """
    if research_mode is None:
        research_mode = RESEARCH_MODE
    
    # Генерируем тестовые данные
    g = generate(n_samples, 10.0, mod_name)
    X, _ = build_xy(g, research_mode)
    
    # Прогрев
    for _ in range(n_warmup):
        _ = model.predict(X, batch_size=n_samples, verbose=0)
    
    # Замеры
    times = []
    for _ in range(n_runs):
        start = time.perf_counter()
        _ = model.predict(X, batch_size=n_samples, verbose=0)
        end = time.perf_counter()
        times.append((end - start) / n_samples)
    
    times = np.array(times)
    return float(np.mean(times)), float(np.std(times))


def print_performance_table(models_dict, mod_name='QPSK'):
    """
    Выводит сравнительную таблицу производительности: FLOPs + время.
    
    models_dict: словарь вида:
    {
        'LS (ZF)': ('classic', 'LS'),
        'MMSE': ('classic', 'MMSE'),
        'ChanEstNet': ('dnn', chan_est_model),
        'DetectorNet': ('dnn', detector_model),
        'JointNet': ('dnn', joint_model),
    }
    """
    print(f"\n{'='*100}")
    print(f"  СРАВНЕНИЕ ПРОИЗВОДИТЕЛЬНОСТИ — {mod_name}")
    print(f"  N_FFT={N_FFT}, N_PILOT={N_PILOT}, N_DATA={N_DATA}")
    print(f"{'='*100}")
    
    results = []
    
    # Измеряем все методы
    for label, (method_type, method_or_model) in models_dict.items():
        if method_type == 'classic':
            # Классический метод
            flops = count_flops_classical(method_or_model, mod_name)
            
            if method_or_model == 'LS':
                det_fn = detect_ls
            elif method_or_model == 'MMSE':
                det_fn = detect_mmse
            else:
                continue
            
            mean_time, std_time = measure_inference_time(
                det_fn, mod_name, snr_dB=10.0, 
                n_warmup=5, n_runs=50, n_samples=500
            )
            
        elif method_type == 'dnn':
            # Нейросетевой метод
            model = method_or_model
            flops, _ = count_flops_dnn(model)
            
            # Определяем research_mode по имени модели
            if 'ChanEst' in model.name:
                research_mode = 'CHANNEL_ESTIMATION'
            elif 'Detector' in model.name:
                research_mode = 'SIGNAL_DETECTION'
            elif 'Joint' in model.name:
                research_mode = 'END_TO_END'
            else:
                research_mode = RESEARCH_MODE
            
            mean_time, std_time = measure_dnn_inference_time(
                model, mod_name, research_mode,
                n_warmup=5, n_runs=50, n_samples=500
            )
        else:
            continue
        
        throughput = 1.0 / mean_time  # символов в секунду
        results.append((label, flops, mean_time, std_time, throughput))
    
    # Сортировка по времени (быстрее → медленнее)
    results.sort(key=lambda x: x[2])
    
    # Базовый уровень — самый быстрый метод
    base_time = results[0][2] if results else 1.0
    base_flops = results[0][1] if results else 1
    
    # Вывод таблицы
    print(f"\n  {'Метод':<30} {'FLOPs':<15} {'Время':<15} {'Throughput':<15} {'Отн. время':<12}")
    print(f"  {'-'*90}")
    for label, flops, mean_time, std_time, throughput in results:
        time_ratio = mean_time / base_time
        print(f"  {label:<30} {format_flops(flops):<15} "
              f"{format_time(mean_time):<15} {throughput:>10,.0f} symb/s  "
              f"{time_ratio:>10.1f}×")
    
    print(f"\n  💡 Интерпретация:")
    max_time = results[-1][2]
    min_time = results[0][2]
    max_flops = results[-1][1]
    min_flops = results[0][1]
    
    print(f"     Самый медленный метод требует в {max_time/min_time:,.1f}× больше времени,")
    print(f"     чем самый быстрый. Разница в FLOPs: {max_flops/min_flops:,.0f}×.")
    print(f"\n     Для IoT-устройств с ограниченным энергопотреблением:")
    print(f"     - Классические методы: {format_time(min_time)} на символ")
    print(f"     - Нейросети: {format_time(max_time)} на символ")
    print(f"     - Это критически важный trade-off: качество vs скорость")
    print(f"\n{'='*100}\n")
    
    return results


def benchmark_all_methods(mod_name, chan_est_model, detector_model, joint_model):
    """
    Полный бенчмарк всех методов для одной модуляции.
    """
    models_dict = {
        'LS (ZF)': ('classic', 'LS'),
        'MMSE': ('classic', 'MMSE'),
        'ChanEstNet': ('dnn', chan_est_model),
        'DetectorNet': ('dnn', detector_model),
        'JointNet': ('dnn', joint_model),
    }
    
    return print_performance_table(models_dict, mod_name)

In [ ]:
# ================================================================
# БЛОК 7: ТЕСТИРОВАНИЕ И ГРАФИКИ
# ================================================================
#
# plot_ber_one_modulation() — BER vs SNR: Oracle + LS + MMSE + DNN.
#   Данные генерируются ОДИН РАЗ для всех методов → объективно.
#   Ось Y — semilogy, нижняя граница по данным (не ниже 1e-7).
#
# plot_ber_one_arch() — одна архитектура, все модуляции.
#
# plot_constellations() — I/Q диаграммы при SNR=0,10,20 дБ.
#   Раскраска по ИСТИННОМУ классу (не принятому): показывает,
#   как шум размывает каждый класс, а не смешение из-за ошибок.
#   alpha=0.03, rasterized=True: тысячи точек → растровый слой.
#
# print_ber_table() — числовая таблица BER с шагом snr_step дБ.
#   Данные генерируются один раз для всех методов на каждом SNR.
# ================================================================

import pandas as pd

_PALETTE = {2: ['#C62828', '#1565C0'], 4: ['#C62828', '#1565C0', '#2E7D32', '#E65100']}

def _palette(n):
    """n≤4: фиксированные цвета. n>4: HSV-круг без повторений."""
    if n in _PALETTE: return _PALETTE[n]
    cmap = plt.colormaps['hsv']
    return [cmap(i / n) for i in range(n)]


def plot_ber_one_modulation(mod_name, models_dict, snr_range=SNR_TEST, n_samples=N_TEST):
    """
    BER vs SNR для одной модуляции: Oracle + LS + MMSE + все DNN.
    models_dict: {label: (type, *args)}
        type='two_stage': (chan_est_model, detector_model)
        type='joint':     (joint_model,)
        type='iterative': (n_iterations,)
        type='classic':   (model,)
    """
    mod = MODULATIONS[mod_name]
    fig, ax = plt.subplots(figsize=(9, 6))
    print(f"  [{mod_name}] Генерация тестовых данных...")
    test_data = [generate(n_samples, float(snr), mod_name) for snr in snr_range]
    all_bers  = []

    # Oracle: MMSE с истинным H — абсолютная нижняя граница
    print(f"  [{mod_name}] Oracle...")
    bers_oracle = []
    for i, snr in enumerate(snr_range):
        g = test_data[i]; H = g['H_true_data']
        sigma2 = 1.0 / (10.0 ** (snr / 10.0))
        W = np.conj(H) / (np.abs(H)**2 + sigma2)
        pred = mod.demodulate((W * g['Y_data']).reshape(-1))
        bers_oracle.append(float(np.mean(pred.reshape(n_samples, -1) != g['bits'])))
    ax.semilogy(snr_range, bers_oracle, 'k--^', lw=2, ms=7, label='Oracle (идеальный H, MMSE)')
    all_bers.append(bers_oracle)

    # LS (ZF)
    print(f"  [{mod_name}] LS...")
    bers_ls = []
    for i, snr in enumerate(snr_range):
        g = test_data[i]
        Hs = np.where(np.abs(g['H_est_data']) < 1e-9, 1e-9+0j, g['H_est_data'])
        pred = mod.demodulate((g['Y_data'] / Hs).reshape(-1))
        bers_ls.append(float(np.mean(pred.reshape(n_samples, -1) != g['bits'])))
    ax.semilogy(snr_range, bers_ls, 'r-o', lw=2, ms=7, label='LS (ZF, DFT-оценка H)')
    all_bers.append(bers_ls)

    # MMSE
    print(f"  [{mod_name}] MMSE...")
    bers_mmse = []
    for i, snr in enumerate(snr_range):
        g = test_data[i]; H = g['H_est_data']
        sigma2 = 1.0 / (10.0 ** (snr / 10.0))
        W = np.conj(H) / (np.abs(H)**2 + sigma2)
        pred = mod.demodulate((W * g['Y_data']).reshape(-1))
        bers_mmse.append(float(np.mean(pred.reshape(n_samples, -1) != g['bits'])))
    ax.semilogy(snr_range, bers_mmse, 'g-s', lw=2, ms=7, label='MMSE (DFT-оценка H)')
    all_bers.append(bers_mmse)

    # DNN-методы и Iterative
    colors_cycle  = ['#2196F3', '#FF9800', '#9C27B0', '#00BCD4', '#FF5722', '#8BC34A']
    markers_cycle = ['D', 'P', 'v', 'X', 'h', '*']
    
    for ci, (label, entry) in enumerate(models_dict.items()):
        c = colors_cycle[ci % len(colors_cycle)]
        mk = markers_cycle[ci % len(markers_cycle)]
        print(f"  [{mod_name}] {label}...")
        
        if entry[0] == 'two_stage':
            _, ce_m, det_m = entry
            bers = compute_ber_two_stage(ce_m, det_m, mod_name, snr_range, n_samples)
        elif entry[0] == 'joint':
            _, jm = entry
            bers = compute_ber_joint(jm, mod_name, snr_range, n_samples)
        elif entry[0] == 'iterative':
            _, n_iters = entry
            bers = compute_ber_iterative(mod_name, snr_range, n_samples, n_iterations=n_iters)
        else:
            _, m = entry
            bers = compute_ber_dnn(m, mod_name, snr_range, n_samples)
            
        ax.semilogy(snr_range, bers, color=c, marker=mk, ls='-', lw=2.5, ms=8, label=label)
        all_bers.append(bers)

    # Умное масштабирование оси Y: по данным, не ниже 1e-7
    valid_bers = np.concatenate([np.array(b)[np.array(b) > 0] for b in all_bers if len(b) > 0])
    y_min = max(10**np.floor(np.log10(np.min(valid_bers))), 1e-7) if len(valid_bers) > 0 else 1e-4
    ax.set_ylim([y_min, 1.0])
    ax.set_xlabel('SNR, дБ (Es/N0)', fontsize=13)
    ax.set_ylabel('BER', fontsize=13)
    ax.set_title(f'BER vs SNR — {mod_name}\n'
                 f'N_FFT={N_FFT}, CP={N_CP}, канал={CHANNEL_MODEL} ({N_TAPS} отв.), '
                 f'сценарий={SCENARIO}', fontsize=12)
    ax.legend(fontsize=10, loc='best')
    ax.grid(True, which='both', alpha=0.4, ls='--')
    ax.set_xlim([snr_range[0]-1, snr_range[-1]+1])
    plt.tight_layout()
    
    fname = f'BER_{mod_name}.png'
    full_path = os.path.join(BASE_SAVE_DIR, fname)
    plt.savefig(full_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  Сохранено: {full_path}")
    plt.close(fig)


def plot_ber_one_arch(arch_name, models_by_mod, mod_names=None,
                      snr_range=SNR_TEST, n_samples=N_TEST):
    """BER одной архитектуры для всех модуляций на одном графике."""
    if mod_names is None: mod_names = list(models_by_mod.keys())
    snr_lin = 10 ** (snr_range / 10.0)
    styl = {'BPSK':('#2196F3','o'),'QPSK':('#4CAF50','s'),
            'QAM16':('#FF9800','^'),'QAM64':('#E91E63','D')}
    fig, ax = plt.subplots(figsize=(10, 6))
    for mn in mod_names:
        mod = MODULATIONS[mn]; c, mk = styl.get(mn, ('#333','o'))
        # Пунктир — теоретическая BER для AWGN (без замираний)
        if ENABLE_THEORY_BER:
            ax.semilogy(snr_range, mod.ber_theory(snr_lin), color=c, ls=':', lw=1.2)
        entry = models_by_mod[mn]
        if   entry[0] == 'two_stage': _, ce_m, det_m = entry; bers = compute_ber_two_stage(ce_m, det_m, mn, snr_range, n_samples)
        elif entry[0] == 'joint':     _, jm = entry;           bers = compute_ber_joint(jm, mn, snr_range, n_samples)
        else:                         _, m = entry;             bers = compute_ber_dnn(m, mn, snr_range, n_samples)
        ax.semilogy(snr_range, bers, color=c, marker=mk, ls='-', lw=2.5, ms=8,
                    label=f'{mn}' + ('  (теория=пунктир)' if ENABLE_THEORY_BER else ''))
    ax.set_xlabel('SNR, дБ', fontsize=13); ax.set_ylabel('BER', fontsize=13)
    ax.set_title(f'BER — {arch_name}  (канал={CHANNEL_MODEL}, сцен={SCENARIO})', fontsize=12)
    ax.legend(fontsize=10); ax.grid(True, which='both', alpha=0.4)
    ax.set_xlim([snr_range[0]-1, snr_range[-1]+1]); ax.set_ylim([1e-4, 1.0])
    plt.tight_layout()
    plt.savefig(os.path.join(BASE_SAVE_DIR, f'BER_arch_{arch_name}.png'), dpi=150, bbox_inches='tight')
    plt.show(); plt.close(fig)


def plot_constellations(mod_name, snr_list=None, n_sym=None):
    """
    I/Q диаграммы созвездий после ZF-эквализации (DFT-оценка H).
    Раскраска по ИСТИННОМУ переданному классу.
    Точки перемешиваются перед отрисовкой, чтобы избежать перекрытия цветов.
    Легенда и окошки с обозначением битов удалены.
    """
    if snr_list is None: snr_list = CONSTELLATION_SNR_LIST
    if n_sym is None: n_sym = CONSTELLATION_N_SYMBOLS

    mod = MODULATIONS[mod_name]
    ideal_pts = mod.constellation
    n_pts = len(ideal_pts)
    n_snr = len(snr_list)
    
    fig, axes = plt.subplots(1, n_snr, figsize=(6 * n_snr, 6))
    if n_snr == 1: axes = [axes]

    # Уникальные цвета для каждой точки созвездия
    colors = _palette(n_pts)

    for ax, snr in zip(axes, snr_list):
        g = generate(n_sym, float(snr), mod_name)
        bits_flat = g['bits'].reshape(n_sym * N_DATA, mod.k)
        tx_syms = mod.modulate(bits_flat)
        
        # Истинный класс = индекс ближайшей точки идеального созвездия
        tx_idx = np.argmin(
            np.abs(tx_syms[:, np.newaxis] - ideal_pts[np.newaxis, :]), axis=1)
        rx_re = np.real(g['eq_symbols'].flatten())
        rx_im = np.imag(g['eq_symbols'].flatten())
        
        # ── Перемешивание точек для корректного z-order ──
        point_colors = np.array([colors[ci] for ci in tx_idx])
        shuffle_idx = np.random.permutation(len(rx_re))
        
        rx_re_shuffled = rx_re[shuffle_idx]
        rx_im_shuffled = rx_im[shuffle_idx]
        colors_shuffled = point_colors[shuffle_idx]
        
        # Рисуем ВСЕ точки одним вызовом в случайном порядке
        ax.scatter(rx_re_shuffled, rx_im_shuffled,
                   color=colors_shuffled, alpha=0.15, s=12,
                   edgecolors='none', rasterized=True)
        # ──────────────────────────────────────────────────

        # Идеальные точки созвездия — чёрные крестики (поверх всего)
        ax.scatter(np.real(ideal_pts), np.imag(ideal_pts),
                   c='black', s=150, zorder=10, marker='+', linewidths=2.0)
        
        ax.axhline(0, color='gray', lw=0.5, alpha=0.4)
        ax.axvline(0, color='gray', lw=0.5, alpha=0.4)
        ax.grid(True, alpha=0.25)
        
        lim = np.max(np.abs(ideal_pts)) * 2.5 + 0.5
        ax.set_xlim([-lim, lim])
        ax.set_ylim([-lim, lim])
        ax.set_aspect('equal')
        ax.set_title(f'SNR = {snr} дБ', fontsize=12, fontweight='bold')
        ax.set_xlabel('In-Phase (I)', fontsize=11)
        ax.set_ylabel('Quadrature (Q)', fontsize=11)
        ax.tick_params(labelsize=9)

    fig.suptitle(
        f'Созвездие {mod_name} после ZF-эквализации\n'
        f'({n_sym}×{N_DATA}={n_sym*N_DATA:,} точек, канал={CHANNEL_MODEL})',
        fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    
    fname = f'constellation_{mod_name}.png'
    full_path = os.path.join(BASE_SAVE_DIR, fname)
    plt.savefig(full_path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print(f"Сохранено: {full_path}")



def plot_constellations_comparison(mod_name, chan_est_model, snr_list=None, n_sym=None):
    """
    Сравнительная визуализация созвездий: Классика vs Нейросеть.
    
    Столбцы:
      1. Классический (LS+ZF) — раскраска по ИСТИННОМУ классу
      2. Классический (LS+ZF) — раскраска по ПРЕДСКАЗАННОМУ классу
      3. Нейросеть (ChanEstNet+ZF) — раскраска по ИСТИННОМУ классу
      4. Нейросеть (ChanEstNet+ZF) — раскраска по ПРЕДСКАЗАННОМУ классу
    
    Каждая из M точек созвездия имеет свой уникальный цвет (HSV-палитра).
    Ошибки детектирования видны как точки «чужого» цвета в кластере.
    """
    if snr_list is None: snr_list = CONSTELLATION_SNR_LIST
    if n_sym is None: n_sym = CONSTELLATION_N_SYMBOLS
    
    mod = MODULATIONS[mod_name]
    ideal_pts = mod.constellation
    n_pts = len(ideal_pts)
    n_snr = len(snr_list)
    
    fig, axes = plt.subplots(n_snr, 4, figsize=(22, 6 * n_snr))
    if n_snr == 1: axes = axes.reshape(1, -1)
    
    # Жесткие заголовки столбцов
    col_titles = [
        'Классический (LS+ZF)\n[Истинные классы]',
        'Классический (LS+ZF)\n[Предсказанные классы]',
        'Нейросеть (ChanEstNet+ZF)\n[Истинные классы]',
        'Нейросеть (ChanEstNet+ZF)\n[Предсказанные классы]'
    ]
    
    # ── Палитра: уникальные цвета для КАЖДОЙ точки созвездия ──
    colors = _palette(n_pts)  # HSV-круг для M=2,4,16,64
    
    fig.suptitle(f'Сравнение приёмников: {mod_name} ({n_sym*N_DATA:,} точек)', 
                 fontsize=18, fontweight='bold', y=0.98)
    
    for row_idx, snr in enumerate(snr_list):
        g = generate(n_sym, float(snr), mod_name)
        
        # ── 1. Классический приёмник (LS+ZF) ──
        classical_symbols = g['eq_symbols'].flatten()
        
        # ── 2. Нейросетевой приёмник (ChanEstNet+ZF) ──
        Yp = g['Y_pilot'] / (g['norm'] if USE_INPUT_NORMALIZATION else 1)
        X_ce = np.hstack([np.real(Yp), np.imag(Yp)])
        if USE_SNR_FEATURE:
            span = float(SNR_TRAIN_MAX - SNR_TRAIN_MIN)
            snr_feat = np.full((n_sym, 1), (snr - SNR_TRAIN_MIN) / span, dtype=np.float32)
            X_ce = np.hstack([X_ce, snr_feat])
        H_dnn_flat = chan_est_model.predict(X_ce.astype(np.float32), verbose=0)
        H_dnn = H_dnn_flat[:, :N_DATA] + 1j * H_dnn_flat[:, N_DATA:]
        if USE_INPUT_NORMALIZATION: H_dnn = H_dnn * g['norm']
        H_safe = np.where(np.abs(H_dnn) < 1e-9, 1e-9 + 0j, H_dnn)
        nn_symbols = (g['Y_data'] / H_safe).flatten()
        
        # ── Определение класса: индекс ближайшей точки созвездия ──
        bits_flat = g['bits'].reshape(n_sym * N_DATA, mod.k)
        tx_syms = mod.modulate(bits_flat)
        
        def get_classes(syms):
            """Класс = индекс ближайшей точки идеального созвездия (0..M-1)."""
            return np.argmin(
                np.abs(syms[:, np.newaxis] - ideal_pts[np.newaxis, :]), axis=1)
        
        true_classes = get_classes(tx_syms)
        classical_pred = get_classes(classical_symbols)
        nn_pred = get_classes(nn_symbols)
        
        # Данные для 4 столбцов: (символы, классы_для_раскраски, показывать_ошибки)
        plots_data = [
            (classical_symbols, true_classes,     False),
            (classical_symbols, classical_pred,   True),
            (nn_symbols,        true_classes,     False),
            (nn_symbols,        nn_pred,          True),
        ]
        
        for col_idx, (syms, classes_to_color, show_errors) in enumerate(plots_data):
            ax = axes[row_idx, col_idx]
            rx_re = np.real(syms)
            rx_im = np.imag(syms)
            
            # ── ИСПРАВЛЕНИЕ Z-ORDER: ПЕРЕМЕШИВАНИЕ ТОЧЕК ──
            # Создаём массив цветов и перемешиваем, чтобы ни один класс
            # не рисовался систематически поверх других
            point_colors = np.array([colors[ci] for ci in classes_to_color])
            shuffle_idx = np.random.permutation(len(rx_re))
            
            rx_re_shuffled = rx_re[shuffle_idx]
            rx_im_shuffled = rx_im[shuffle_idx]
            colors_shuffled = point_colors[shuffle_idx]
            
            # Рисуем ВСЕ точки одним вызовом в случайном порядке
            # alpha=0.15 чтобы 64 цвета были различимы
            ax.scatter(rx_re_shuffled, rx_im_shuffled, color=colors_shuffled,
                       alpha=0.15, s=12, edgecolors='none', rasterized=True)
            
            # ── Идеальные точки созвездия — чёрные крестики (поверх) ──
            ax.scatter(np.real(ideal_pts), np.imag(ideal_pts),
                       c='black', s=120, zorder=10, marker='+', linewidths=2.0)
            
            # ── Информационный блок с SER ──
            if show_errors:
                error_mask = (classes_to_color != true_classes)
                ser_val = np.mean(error_mask)
                box_text = f'SER = {ser_val:.1%}'
                box_color = '#d62728' if ser_val > 0.20 else '#2ca02c'
            else:
                box_text = 'Истинные классы'
                box_color = 'gray'
            
            ax.text(0.05, 0.95, box_text, transform=ax.transAxes,
                    fontsize=11, fontweight='bold',
                    verticalalignment='top',
                    bbox=dict(boxstyle='round,pad=0.4',
                              facecolor='white', alpha=0.9, edgecolor=box_color))
            
            # Заголовки столбцов (только для первой строки)
            if row_idx == 0:
                ax.set_title(col_titles[col_idx], fontsize=14,
                             fontweight='bold', pad=15, color='#1565C0')
            
            # Подписи строк (SNR) и осей
            if col_idx == 0:
                ax.set_ylabel(f'SNR = {snr} дБ\n\nQuadrature (Q)',
                              fontsize=12, fontweight='bold', labelpad=10)
            if row_idx == n_snr - 1:
                ax.set_xlabel('In-Phase (I)', fontsize=11)
            
            # Сетка и масштаб
            ax.axhline(0, color='gray', lw=0.5, alpha=0.4)
            ax.axvline(0, color='gray', lw=0.5, alpha=0.4)
            ax.grid(True, alpha=0.25)
            lim = np.max(np.abs(ideal_pts)) * 2.5 + 0.5
            ax.set_xlim([-lim, lim])
            ax.set_ylim([-lim, lim])
            ax.set_aspect('equal')
            ax.tick_params(labelsize=9)
    
    plt.tight_layout(rect=[0.03, 0.02, 1, 0.95])
    
    fname = f'constellation_comparison_{mod_name}.png'
    full_path = os.path.join(BASE_SAVE_DIR, fname)
    plt.savefig(full_path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print(f"Сохранено: {full_path}")

def plot_received_signal_raw(mod_name, snr=10, n_sym=200):
    """
    Показывает "сырой" принятый сигнал до эквализации.
    Это то, что реально "видит" приёмник на входе.
    """
    g = generate(n_sym, snr, mod_name)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Левый график: Y_data (принятый сигнал с каналом и шумом)
    Y_data_flat = g['Y_data'].flatten()
    axes[0].scatter(np.real(Y_data_flat), np.imag(Y_data_flat),
                   s=10, alpha=0.5, c='blue')
    axes[0].set_title(f'Принятый сигнал Y (до эквализации)\nSNR = {snr} dB')
    axes[0].set_xlabel('Re')
    axes[0].set_ylabel('Im')
    axes[0].grid(True, alpha=0.3)
    axes[0].set_xlim(-3, 3)
    axes[0].set_ylim(-3, 3)
    
    # Правый график: H_true (истинный канал)
    H_true_flat = g['H_true_data'].flatten()
    axes[1].scatter(np.real(H_true_flat), np.imag(H_true_flat),
                   s=10, alpha=0.5, c='red')
    axes[1].set_title(f'Истинный канал H (искажения)')
    axes[1].set_xlabel('Re')
    axes[1].set_ylabel('Im')
    axes[1].grid(True, alpha=0.3)
    axes[1].set_xlim(-2, 2)
    axes[1].set_ylim(-2, 2)
    
    plt.tight_layout()
    filename = f'{BASE_SAVE_DIR}/received_signal_raw_{mod_name}_snr{snr}.png'
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    print(f"Сохранено: {filename}")
    plt.show()


def print_ber_table(mod_name, chan_est_model, detector_model, joint_model,
                    snr_step=5, n_samples=N_TEST, n_iterations=3):
    """
    Числовая таблица BER. Для каждого SNR данные генерируются ОДИН РАЗ
    и используются для Oracle, LS, MMSE, DNN Two-Stage, DNN JointNet, Iterative.
    Сохраняет таблицу в CSV для последующего анализа.
    """
    mod = MODULATIONS[mod_name]
    snr_range_stepped = np.arange(SNR_TEST[0], SNR_TEST[-1] + 1, snr_step)
    print(f"\n{'='*100}")
    print(f"  ТАБЛИЦА BER — {mod_name}  (шаг SNR = {snr_step} дБ)")
    print(f"{'='*100}")
    print(f"  Вычисление...")
    
    results = []
    for snr in snr_range_stepped:
        g = generate(n_samples, float(snr), mod_name)
        sigma2 = 1.0 / (10.0 ** (snr / 10.0))
        
        # Oracle
        H = g['H_true_data']
        W = np.conj(H) / (np.abs(H)**2 + sigma2)
        pred = mod.demodulate((W * g['Y_data']).reshape(-1))
        ber_oracle = float(np.mean(pred.reshape(n_samples, -1) != g['bits']))
        
        # LS
        Hs = np.where(np.abs(g['H_est_data']) < 1e-9, 1e-9+0j, g['H_est_data'])
        pred = mod.demodulate((g['Y_data'] / Hs).reshape(-1))
        ber_ls = float(np.mean(pred.reshape(n_samples, -1) != g['bits']))
        
        # MMSE
        H = g['H_est_data']
        W = np.conj(H) / (np.abs(H)**2 + sigma2)
        pred = mod.demodulate((W * g['Y_data']).reshape(-1))
        ber_mmse = float(np.mean(pred.reshape(n_samples, -1) != g['bits']))
        
        # DNN two-stage
        ber_2s = float(np.mean(compute_ber_two_stage(
            chan_est_model, detector_model, mod_name, [snr], n_samples)))
        
        # DNN JointNet
        ber_jn = float(np.mean(compute_ber_joint(
            joint_model, mod_name, [snr], n_samples)))
        
        # Iterative Cancellation
        ber_iter = float(np.mean(compute_ber_iterative(
            mod_name, [snr], n_samples, n_iterations=n_iterations)))
        
        results.append({
            'SNR (дБ)': int(snr),
            'Oracle': ber_oracle,
            'LS (ZF)': ber_ls,
            'MMSE': ber_mmse,
            'DNN 2-Stage': ber_2s,
            'DNN JointNet': ber_jn,
            f'Iterative ({n_iterations})': ber_iter
        })
    
    df = pd.DataFrame(results)
    
    def fmt_ber(x):
        if x < 1e-4:
            return f"{x:.2e}"
        elif x < 1e-2:
            return f"{x:.5f}"
        else:
            return f"{x:.4f}"
    
    df_d = df.copy()
    for col in df_d.columns:
        if col != 'SNR (дБ)':
            df_d[col] = df_d[col].apply(fmt_ber)
    
    print()
    print(df_d.to_string(index=False))
    print(f"{'='*100}\n")
    
    csv_name = f'BER_table_{mod_name}.csv'
    full_csv_path = os.path.join(BASE_SAVE_DIR, csv_name)
    df.to_csv(full_csv_path, index=False, encoding='utf-8')
    print(f"  Таблица сохранена: {full_csv_path}\n")



print("✓ Блок 7 готов.")

In [ ]:
# ================================================================
# БЛОК 8: ЗАПУСК ЭКСПЕРИМЕНТОВ + БЕНЧМАРК ПРОИЗВОДИТЕЛЬНОСТИ
# ================================================================
import time
import math

# ────────────────────────────────────────────────────────────────
# GUI запуск эксперментов
# ────────────────────────────────────────────────────────────────

def run_experiment_with_gui(outputs, gui):
    """
    Запускает эксперимент с настройками из GUI
    """
    global chan_est_models, detector_models, joint_models
    
    chan_est_models = {}
    detector_models = {}
    joint_models = {}
    
    print("=" * 62)
    print("ЗАПУСК ЭКСПЕРИМЕНТА")
    print("=" * 62)
    print(f"  CHANNEL_MODEL : {CHANNEL_MODEL}  (N_TAPS={N_TAPS})")
    print(f"  SCENARIO      : {SCENARIO}")
    print(f"  MOD_TO_RUN    : {MOD_TO_RUN}")
    print(f"  SAVE DIR      : {BASE_SAVE_DIR}/")
    
    # ── Шаги 1-3: Обучение всех модуляций ──
    for mod_name in MOD_TO_RUN:
        if gui.stop_requested:
            print("\n⏹️ Остановлено пользователем")
            return
            
        print(f"\n{'━'*62}  {mod_name}  {'━'*62}")
        
        # Шаг 1: ChanEstNet
        m_ce, loaded = load_model_if_exists('ChanEstNet', mod_name, 'step1')
        if not loaded:
            m_ce, h_ce = train_chan_est(mod_name)
            fig, ax = plt.subplots(figsize=(7, 3))
            ax.plot(h_ce, lw=2, color='#2196F3')
            ax.set_title(f'ChanEstNet {mod_name} — MSE')
            ax.set_xlabel('Эпоха')
            ax.grid(True, alpha=0.4)
            plt.tight_layout()
            plt.savefig(os.path.join(BASE_SAVE_DIR, f'train_ChanEst_{mod_name}.png'), dpi=120)
            plt.show()
            plt.close(fig)
        chan_est_models[mod_name] = m_ce
        
        # Шаг 2: DetectorNet
        m_det, loaded = load_model_if_exists('DetectorNet', mod_name, 'step2')
        if not loaded:
            m_det, h_det = train_detector(mod_name, m_ce)
            fig, ax = plt.subplots(figsize=(7, 3))
            ax.plot(h_det, lw=2, color='#FF9800')
            ax.set_title(f'DetectorNet {mod_name} — BCE/MSE')
            ax.set_xlabel('Эпоха')
            ax.grid(True, alpha=0.4)
            plt.tight_layout()
            plt.savefig(os.path.join(BASE_SAVE_DIR, f'train_Det_{mod_name}.png'), dpi=120)
            plt.show()
            plt.close(fig)
        detector_models[mod_name] = m_det
        
        # Шаг 3: JointNet
        m_j, loaded = load_model_if_exists('JointNet', mod_name, 'step3')
        if not loaded:
            m_j, h_j = train_joint(mod_name, chan_est_model=m_ce)
            fig, ax = plt.subplots(figsize=(7, 3))
            ax.plot(h_j, lw=2, color='#9C27B0')
            ax.set_title(f'JointNet {mod_name} — total loss')
            ax.set_xlabel('Эпоха')
            ax.grid(True, alpha=0.4)
            plt.tight_layout()
            plt.savefig(os.path.join(BASE_SAVE_DIR, f'train_Joint_{mod_name}.png'), dpi=120)
            plt.show()
            plt.close(fig)
        joint_models[mod_name] = m_j
    
    print("\n✓ Все модели готовы.")
    
    # ── Шаг 4: Созвездия (если включено) ──
    if outputs['constellations']:
        print("\n► Созвездия (качественная проверка системы)...")
        for mod_name in MOD_TO_RUN:
            if gui.stop_requested:
                break
            plot_constellations(mod_name)
    
    # ── Шаг 4.1: Сырой сигнал (если включено) ──
    if outputs['raw_signal']:
        print("\n► Сырой сигнал до эквализации...")
        for mod_name in ['BPSK', 'QPSK']:
            if gui.stop_requested:
                break
            for snr in [5, 15, 25]:
                plot_received_signal_raw(mod_name, snr=snr, n_sym=200)
    
    # ── Шаг 4.5: Сравнение созвездий (если включено) ──
    if outputs['constellations_comparison']:
        print("\n► Расширенное сравнение созвездий...")
        for mod_name in MOD_TO_RUN:
            if gui.stop_requested:
                break
            plot_constellations_comparison(
                mod_name,
                chan_est_models[mod_name],
                snr_list=[0, 10, 20],
                n_sym=500
            )
    
    # ── Шаг 4.6: Бенчмарк производительности (если включено) ──
    if outputs['benchmark']:
        print("\n► Бенчмарк производительности (FLOPs + время)...")
        for mod_name in MOD_TO_RUN:
            if gui.stop_requested:
                break
            benchmark_all_methods(
                mod_name,
                chan_est_models[mod_name],
                detector_models[mod_name],
                joint_models[mod_name]
            )
    
    # ── Шаг 5: BER-кривые + таблицы (если включено) ──
    if outputs['ber_curves'] or outputs['ber_tables']:
        print("\n► BER-кривые...")
        for mod_name in MOD_TO_RUN:
            if gui.stop_requested:
                break
                
            dnn_entries = {
                'DNN Two-Stage': ('two_stage', chan_est_models[mod_name], detector_models[mod_name]),
                'DNN JointNet': ('joint', joint_models[mod_name]),
                'Iterative (3)': ('iterative', 3),
            }
            
            if outputs['ber_curves']:
                plot_ber_one_modulation(mod_name, dnn_entries, snr_range=SNR_TEST, n_samples=N_TEST)
            
            if outputs['ber_tables']:
                print_ber_table(
                    mod_name=mod_name,
                    chan_est_model=chan_est_models[mod_name],
                    detector_model=detector_models[mod_name],
                    joint_model=joint_models[mod_name],
                    snr_step=5,
                    n_samples=N_TEST
                )
    
    if not gui.stop_requested:
        print("\n✓ Все эксперименты завершены!")
    
    # Очистка памяти
    cleanup_gpu_memory()

    
# ────────────────────────────────────────────────────────────────
# ФУНКЦИИ ДЛЯ БЕНЧМАРКА (FLOPs + ВРЕМЯ ВЫПОЛНЕНИЯ)
# ────────────────────────────────────────────────────────────────

def _flops_fft(N):
    """Complex FFT: ~5·N·log₂(N) операций (Split-radix)."""
    if N <= 1: return 0
    return int(5 * N * math.log2(N))

def count_flops_classical(method='LS', mod_name='QPSK'):
    """Подсчёт FLOPs для классического детектора на один OFDM-символ."""
    mod = MODULATIONS[mod_name]
    M = mod.k  # bits per symbol
    total = 0
    # DFT-based оценка канала
    total += _flops_fft(N_PILOT) + _flops_fft(N_FFT)
    # Эквализация N_DATA поднесущих
    if method == 'LS':
        total += N_DATA * 12  # комплексное деление
    elif method == 'MMSE':
        total += N_DATA * (3 + 1 + 6 + 12)  # |H|² + σ² + conj(H)*Y + div
    # ML-детектирование: M евклидовых расстояний на поднесущую
    total += N_DATA * mod.M * 7
    return int(total)


def count_flops_dnn(model):
    """Подсчёт FLOPs для нейросети Keras на один forward pass."""
    total_flops = 0
    for layer in model.layers:
        lt = type(layer).__name__
        
        # Пропускаем входной слой, у него нет "входа" от предыдущего слоя
        if lt == 'InputLayer':
            continue
            
        # ── Безопасное получение input_shape (совместимость Keras 2 и Keras 3) ──
        try:
            # Стандартный способ (работает в большинстве случаев)
            ish = layer.input_shape
        except AttributeError:
            try:
                # Фоллбэк для новых версий Keras 3 / TF 2.16+
                inp = layer.input
                ish = tuple(inp.shape) if not isinstance(inp, list) else [tuple(x.shape) for x in inp]
            except Exception:
                # Если форму узнать не удалось (например, специфичный кастомный слой), пропускаем
                continue 

        if lt == 'Dense':
            in_sz = ish[-1] if not isinstance(ish, list) else ish[0][-1]
            out_sz = layer.units
            total_flops += 2 * in_sz * out_sz
            if layer.use_bias: total_flops += out_sz
            
        elif lt == 'Conv1D':
            _ish = ish[0] if isinstance(ish, list) else ish
            seq_len, c_in = _ish[1], _ish[2]
            c_out, k = layer.filters, layer.kernel_size[0]
            total_flops += 2 * seq_len * k * c_in * c_out
            if layer.use_bias: total_flops += seq_len * c_out
            
        elif lt == 'BatchNormalization':
            _ish = ish[0] if isinstance(ish, list) else ish
            n_el = 1
            for d in _ish[1:]:
                if d: n_el *= d
            total_flops += 4 * n_el
            
        elif lt == 'ReLU':
            _ish = ish[0] if isinstance(ish, list) else ish
            n_el = 1
            for d in _ish[1:]:
                if d: n_el *= d
            total_flops += n_el
            
        elif lt == 'Add':
            _ish = ish[0] if isinstance(ish, list) else ish
            n_el = 1
            for d in _ish[1:]:
                if d: n_el *= d
            total_flops += n_el
            
    return int(total_flops)

def format_flops(n):
    if n >= 1e9: return f"{n/1e9:.2f} GFLOPs"
    elif n >= 1e6: return f"{n/1e6:.2f} MFLOPs"
    elif n >= 1e3: return f"{n/1e3:.2f} kFLOPs"
    else: return f"{n} FLOPs"


def format_time(seconds):
    if seconds >= 1.0: return f"{seconds:.3f} s"
    elif seconds >= 1e-3: return f"{seconds*1e3:.2f} ms"
    elif seconds >= 1e-6: return f"{seconds*1e6:.1f} µs"
    else: return f"{seconds*1e9:.0f} ns"


def measure_time(detector_fn, mod_name, snr_dB=10.0, n_warmup=5, n_runs=50, n_samples=500):
    """Измеряет среднее время выполнения детектора на один OFDM-символ."""
    g = generate(n_samples, snr_dB, mod_name)
    for _ in range(n_warmup):
        _ = detector_fn(g, mod_name, snr_dB)
    times = []
    for _ in range(n_runs):
        start = time.perf_counter()
        _ = detector_fn(g, mod_name, snr_dB)
        end = time.perf_counter()
        times.append((end - start) / n_samples)
    return float(np.mean(times)), float(np.std(times))


def measure_dnn_time(model, mod_name, research_mode, chan_est_model=None,
                     n_warmup=5, n_runs=50, n_samples=500):
    """
    Измеряет время выполнения нейросетевого детектора.
    Корректно готовит входные данные для каждой архитектуры.
    """
    g = generate(n_samples, 10.0, mod_name)
    
    if research_mode == 'CHANNEL_ESTIMATION':
        # ChanEstNet: стандартный build_xy
        X, _ = build_xy(g, research_mode)
        
    elif research_mode == 'SIGNAL_DETECTION':
        # DetectorNet: нужен H_dnn от ChanEstNet + _make_detector_input
        Xp, _ = build_xy(g, 'CHANNEL_ESTIMATION')
        H_dnn_flat = chan_est_model.predict(Xp, batch_size=n_samples, verbose=0)
        X = _make_detector_input(g, H_dnn_flat, 10.0, n_samples)
        
    elif research_mode == 'END_TO_END':
        # JointNet: специальный формат [Yp_norm, Yd_norm, snr]
        extra = 1 if USE_SNR_FEATURE else 0
        Yp = g['Y_pilot'] / (g['norm'] if USE_INPUT_NORMALIZATION else 1)
        Yd = g['Y_data'] / (g['norm'] if USE_INPUT_NORMALIZATION else 1)
        parts = [np.real(Yp).astype(np.float32), np.imag(Yp).astype(np.float32),
                 np.real(Yd).astype(np.float32), np.imag(Yd).astype(np.float32)]
        if extra > 0:
            span = float(SNR_TRAIN_MAX - SNR_TRAIN_MIN)
            parts.append(np.full((n_samples, 1), (10.0 - SNR_TRAIN_MIN) / span, dtype=np.float32))
        X = np.hstack(parts).astype(np.float32)
        
    else:
        X, _ = build_xy(g, research_mode)
    
    # Прогрев
    for _ in range(n_warmup):
        _ = model.predict(X, batch_size=n_samples, verbose=0)
    
    # Замеры
    times = []
    for _ in range(n_runs):
        start = time.perf_counter()
        _ = model.predict(X, batch_size=n_samples, verbose=0)
        end = time.perf_counter()
        times.append((end - start) / n_samples)
    
    return float(np.mean(times)), float(np.std(times))


def benchmark_all_methods(mod_name, chan_est_model, detector_model, joint_model):
    """Полный бенчмарк всех методов для одной модуляции."""
    print(f"\n{'='*100}")
    print(f"  СРАВНЕНИЕ ПРОИЗВОДИТЕЛЬНОСТИ — {mod_name}")
    print(f"  N_FFT={N_FFT}, N_PILOT={N_PILOT}, N_DATA={N_DATA}")
    print(f"{'='*100}")

    results = []

    # Классические методы
    for method_name, det_fn in [('LS (ZF)', detect_ls), ('MMSE', detect_mmse)]:
        flops = count_flops_classical(method_name.split()[0], mod_name)
        mean_t, std_t = measure_time(det_fn, mod_name)
        results.append((method_name, flops, mean_t, std_t))

    # ChanEstNet
    flops = count_flops_dnn(chan_est_model)
    mean_t, std_t = measure_dnn_time(chan_est_model, mod_name, 'CHANNEL_ESTIMATION')
    results.append(('ChanEstNet', flops, mean_t, std_t))
    
    # DetectorNet (нужен chan_est_model для подготовки входа)
    flops = count_flops_dnn(detector_model)
    mean_t, std_t = measure_dnn_time(detector_model, mod_name, 'SIGNAL_DETECTION', chan_est_model)
    results.append(('DetectorNet', flops, mean_t, std_t))
    
    # JointNet
    flops = count_flops_dnn(joint_model)
    mean_t, std_t = measure_dnn_time(joint_model, mod_name, 'END_TO_END')
    results.append(('JointNet', flops, mean_t, std_t))

    # Сортировка по времени
    results.sort(key=lambda x: x[2])
    base_time = results[0][2] if results else 1.0
    base_flops = results[0][1] if results else 1

    print(f"\n  {'Метод':<30} {'FLOPs':<15} {'Время':<15} {'Throughput':<15} {'Отн. время':<12}")
    print(f"  {'-'*90}")
    for label, flops, mean_t, std_t in results:
        throughput = 1.0 / mean_t if mean_t > 0 else 0
        ratio = mean_t / base_time if base_time > 0 else 0
        print(f"  {label:<30} {format_flops(flops):<15} "
              f"{format_time(mean_t):<15} {throughput:>10,.0f} symb/s  "
              f"{ratio:>10.1f}×")

    max_t = results[-1][2]
    min_t = results[0][2]
    max_f = results[-1][1]
    min_f = results[0][1]
    print(f"\n   Самый медленный метод: {max_t/min_t:,.1f}× по времени, {max_f/min_f:,.0f}× по FLOPs")
    print(f"{'='*100}\n")





# ────────────────────────────────────────────────────────────────
# ОСНОВНОЙ ЦИКЛ ЭКСПЕРИМЕНТА
# ────────────────────────────────────────────────────────────────

def cleanup_gpu_memory(verbose=True):
    """Очистка TF-сессии после завершения всех экспериментов."""
    if verbose: print("\n► Очистка GPU-памяти...")
    keras.backend.clear_session(); gc.collect()
    if verbose:
        print("  Keras backend session очищена.")
        print("  Полный возврат VRAM — только при Kernel → Restart.")


chan_est_models = {}
detector_models = {}
joint_models    = {}

print("=" * 62)
print("ЗАПУСК ЭКСПЕРИМЕНТА")
print("=" * 62)
print(f"  CHANNEL_MODEL : {CHANNEL_MODEL}  (N_TAPS={N_TAPS})")
print(f"  SCENARIO      : {SCENARIO}")
print(f"  MOD_TO_RUN    : {MOD_TO_RUN}")
print(f"  SAVE DIR      : {BASE_SAVE_DIR}/")

# ── Шаги 1-3: Обучение всех модуляций ───────────────────────────
for mod_name in MOD_TO_RUN:
    print(f"\n{'━'*62}  {mod_name}  {'━'*62}")

    # Шаг 1: ChanEstNet
    m_ce, loaded = load_model_if_exists('ChanEstNet', mod_name, 'step1')
    if not loaded:
        m_ce, h_ce = train_chan_est(mod_name)
        fig, ax = plt.subplots(figsize=(7, 3))
        ax.plot(h_ce, lw=2, color='#2196F3')
        ax.set_title(f'ChanEstNet {mod_name} — MSE'); ax.set_xlabel('Эпоха')
        ax.grid(True, alpha=0.4); plt.tight_layout()
        plt.savefig(os.path.join(BASE_SAVE_DIR, f'train_ChanEst_{mod_name}.png'), dpi=120)
        plt.show(); plt.close(fig)
    chan_est_models[mod_name] = m_ce

    # Шаг 2: DetectorNet
    m_det, loaded = load_model_if_exists('DetectorNet', mod_name, 'step2')
    if not loaded:
        m_det, h_det = train_detector(mod_name, m_ce)
        fig, ax = plt.subplots(figsize=(7, 3))
        ax.plot(h_det, lw=2, color='#FF9800')
        ax.set_title(f'DetectorNet {mod_name} — BCE/MSE'); ax.set_xlabel('Эпоха')
        ax.grid(True, alpha=0.4); plt.tight_layout()
        plt.savefig(os.path.join(BASE_SAVE_DIR, f'train_Det_{mod_name}.png'), dpi=120)
        plt.show(); plt.close(fig)
    detector_models[mod_name] = m_det

    # Шаг 3: JointNet
    m_j, loaded = load_model_if_exists('JointNet', mod_name, 'step3')
    if not loaded:
        m_j, h_j = train_joint(mod_name, chan_est_model=m_ce)
        fig, ax = plt.subplots(figsize=(7, 3))
        ax.plot(h_j, lw=2, color='#9C27B0')
        ax.set_title(f'JointNet {mod_name} — total loss'); ax.set_xlabel('Эпоха')
        ax.grid(True, alpha=0.4); plt.tight_layout()
        plt.savefig(os.path.join(BASE_SAVE_DIR, f'train_Joint_{mod_name}.png'), dpi=120)
        plt.show(); plt.close(fig)
    joint_models[mod_name] = m_j

print("\n✓ Все модели готовы.")



# ── Шаг 4: Созвездия ─────────────────────────────────────────────
print("\n► Созвездия (качественная проверка системы)...")
for mod_name in MOD_TO_RUN:
    plot_constellations(mod_name)

# ── Шаг 4.1: Сырой принятый сигнал (до эквализации) ──────────────
# Показывает, что реально "видит" приёмник на входе:
#   - Левая панель: принятый сигнал Y с наложением канала и шума
#   - Правая панель: истинная частотная характеристика канала H
# Особенно полезно при включённой нелинейности PA (USE_PA_NONLINEARITY=True):
#   видно, как нелинейность + канал + шум искажают идеальные точки созвездия
print("\n► Сырой сигнал до эквализации (визуализация искажений канала)...")
for mod_name in ['BPSK', 'QPSK']:  # только для простых модуляций для читаемости
    for snr in [5, 15, 25]:        # разные уровни шума
        plot_received_signal_raw(mod_name, snr=snr, n_sym=200)


    
# # ── Шаг 4.5: Расширенное сравнение созвездий ─────────────────────
# print("\n► Расширенное сравнение созвездий (классика vs нейросеть)...")
# for mod_name in MOD_TO_RUN:
#     if mod_name in ['BPSK', 'QPSK', 'QAM16', 'QAM64']:  # Только для простых модуляций (M≤4)
#         plot_constellations_comparison(
#             mod_name, 
#             chan_est_models[mod_name],  # ← Передаём САМУ модель, а не словарь!
#             snr_list=[0, 10, 20],
#             n_sym=500
#         )


# ── Шаг 4.5: Сравнительная визуализация ──────────────────────
print("\n► Сравнение приёмников (классика vs нейросеть)...")
for mod_name in MOD_TO_RUN:
    plot_constellations_comparison(mod_name, chan_est_models[mod_name])


# ── Шаг 4.6: Бенчмарк производительности ─────────────────────────
print("\n► Бенчмарк производительности (FLOPs + время)...")
for mod_name in MOD_TO_RUN:
    benchmark_all_methods(
        mod_name,
        chan_est_models[mod_name],
        detector_models[mod_name],
        joint_models[mod_name]
    )


# ── Шаг 5: BER-кривые + таблицы ──────────────────────────────────
print("\n► BER-кривые...")
for mod_name in MOD_TO_RUN:
    dnn_entries = {
        'DNN Two-Stage': ('two_stage', chan_est_models[mod_name], detector_models[mod_name]),
        'DNN JointNet': ('joint', joint_models[mod_name]),
        'Iterative (3)': ('iterative', 3),  # <-- ДОБАВИЛИ (3 итерации)
    }
    plot_ber_one_modulation(mod_name, dnn_entries, snr_range=SNR_TEST, n_samples=N_TEST)
    print_ber_table(
        mod_name=mod_name,
        chan_est_model=chan_est_models[mod_name],
        detector_model=detector_models[mod_name],
        joint_model=joint_models[mod_name],
        snr_step=5,
        n_samples=N_TEST
    )

print("\n✓ Все эксперименты завершены!")

# ── Шаг 6: Очистка памяти ────────────────────────────────────────
cleanup_gpu_memory()